# Ablasi normalisasi baseline sensor

Batch 06 Exp02 memisahkan pengaruh transformasi RMSSD dan eye menggunakan
empat peserta, 671 window Train, 99 window validation, serta fold subset eye
Exp01 yang sama. Semua varian memakai sepuluh fitur core dan Extra Trees
200 trees dengan parameter tetap. Tidak ada tuning atau evaluasi Test.

Empat varian: raw, RMSSD-only, eye-only, dan gabungan. Kata only mengacu pada
normalisasi yang diaktifkan; fitur sensor lain tetap masuk classifier.

## Mount dan input

Cukup notebook ini serta tiga CSV di root Drive yang sama dengan batch sebelumnya.
Snapshot cohort, baseline, fold, dan bukti sumber sudah tertanam; tidak perlu
output Exp01, file Python pendamping, atau sesi mentah di Drive. Runtime CPU cukup.
Hash CSV harus sesuai run Exp01 agar pembanding benar-benar identik. Jika data
berubah, notebook berhenti dan tidak mencari split pengganti secara otomatis.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
DATASET_ROOT = PROJECT_ROOT / 'Dataset/v2_session_phase'
OUTPUT_ROOT = PROJECT_ROOT / 'models/batch-06/outputs/exp02'

## Lingkungan

Model memakai CPU. Paket yang belum ada dipasang otomatis. Versi aktual dicatat;
kontrol raw dan gabungan dihitung ulang dalam run ini. Perbedaan terhadap Exp01
dilaporkan, tanpa mengganti seed atau parameter untuk memaksakan kecocokan.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'numpy': 'numpy', 'pandas': 'pandas>=2.0', 'sklearn': 'scikit-learn>=1.4',
            'joblib': 'joblib', 'matplotlib': 'matplotlib', 'threadpoolctl': 'threadpoolctl'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

## Definisi sensor dan fitting

Definisi dasar sama dengan Exp01; cell ini hanya mendefinisikan fungsi, tanpa
menjalankan eksperimen Exp01. Seluruh scaler dihitung dari fitting fold.

In [ ]:
"""Definitions embedded in the standalone Batch 06 Colab notebook.

Only sensor CSVs are runtime inputs. Source timing evidence is supplied as a
small, separately checksummed reference, never as predictor columns.
"""

import hashlib
import importlib.metadata
import json
import platform
import uuid
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut
from sklearn.preprocessing import RobustScaler
from threadpoolctl import threadpool_limits


VERSION = "batch06-baseline-v1"
LABELS = ("Low", "Medium", "High")
CORE = [
    "rmssd_ms", "sdnn_ms", "blink_rate_per_minute", "ear_left_mean",
    "ear_left_std", "ear_right_mean", "ear_right_std",
    "iris_size_estimate_mean", "iris_size_estimate_std", "iris_size_delta_mean",
]
FAMILIES = ("logistic_regression", "extra_trees", "dummy")
MODELS = {
    "logistic_regression": {"C": 1.0, "max_iter": 3000},
    "extra_trees": {
        "n_estimators": 200, "min_samples_leaf": 5,
        "max_features": 0.8, "random_state": 42, "n_jobs": 2,
    },
    "dummy": {"strategy": "most_frequent"},
}


def sha256_file(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def bool_values(values):
    return values.astype(str).str.lower().isin(("true", "1"))


def validate_frame(frame, window):
    required = CORE + [
        "window_id", "session_id", "participant_id", "window_start", "window_end",
        "window_seconds", "cognitive_load_label", "study_phase", "quality_tier",
        "baseline_rmssd_ms", "baseline_coverage", "baseline_usable",
    ]
    missing = set(required) - set(frame)
    if missing:
        raise ValueError(f"CSV {window}s kekurangan kolom: {sorted(missing)}")
    result = frame.copy()
    if result.empty or result.window_id.duplicated().any():
        raise ValueError(f"CSV {window}s kosong atau window_id berulang.")
    for col in ("window_id", "session_id", "participant_id"):
        if result[col].isna().any() or result[col].astype(str).str.strip().eq("").any():
            raise ValueError(f"Identitas {col} kosong.")
        result[col] = result[col].astype(str)
    if not set(result.cognitive_load_label).issubset(LABELS):
        raise ValueError("Target harus Low, Medium, atau High.")
    if not set(result.quality_tier).issubset(("good", "acceptable")):
        raise ValueError("Gunakan good_acceptable.csv, bukan all_data.csv.")
    for col in ("window_start", "window_end"):
        if not result[col].map(lambda v: pd.Timestamp(v).tzinfo is not None).all():
            raise ValueError("Timestamp harus menyimpan zona waktu.")
    result["_start"] = pd.to_datetime(result.window_start, utc=True).dt.as_unit("ns").astype("int64")
    result["_end"] = pd.to_datetime(result.window_end, utc=True).dt.as_unit("ns").astype("int64")
    if not np.allclose((result._end - result._start) / 1e9, window, rtol=0, atol=1e-6):
        raise ValueError("Durasi timestamp berbeda dari panjang window.")
    if not pd.to_numeric(result.window_seconds, errors="raise").eq(window).all():
        raise ValueError("window_seconds tidak sesuai folder.")
    x = result[CORE].to_numpy(float)
    if not np.isfinite(x).all():
        raise ValueError("Core berisi missing/infinity; tidak ada imputasi.")
    nonnegative = [c for c in CORE if c != "iris_size_delta_mean"]
    if result[nonnegative].lt(0).any().any() or result.iris_size_estimate_mean.le(0).any():
        raise ValueError("Fitur sensor berada di luar rentang yang diizinkan.")
    if not result[["ear_left_mean", "ear_right_mean"]].le(1).all().all():
        raise ValueError("EAR lebih dari satu.")
    if result.groupby("session_id").participant_id.nunique().gt(1).any():
        raise ValueError("Satu sesi dipetakan ke beberapa orang.")
    result["_key"] = result.session_id + "|" + result._end.astype(str)
    if result._key.duplicated().any():
        raise ValueError("Endpoint sesi berulang.")
    result["_y"] = result.cognitive_load_label.map(dict(zip(LABELS, range(3)))).astype(int)
    return result.sort_values(["session_id", "_end"]).reset_index(drop=True)


def load_views(dataset_root):
    frames, hashes, paths = {}, {}, {}
    for window in (15, 25, 45):
        path = Path(dataset_root) / f"window_{window}s_step_5s/good_acceptable.csv"
        if not path.is_file():
            raise FileNotFoundError(f"Dataset tidak ditemukan: {path}")
        hashes[str(window)] = sha256_file(path)
        paths[str(window)] = str(path)
        frames[window] = validate_frame(pd.read_csv(path, float_precision="round_trip"), window)
    joined = pd.concat([f[["_key", "session_id", "participant_id", "_y"]] for f in frames.values()])
    if joined.groupby("_key")._y.nunique().gt(1).any():
        raise ValueError("Target endpoint berbeda antarwindow.")
    if joined.groupby("session_id").participant_id.nunique().gt(1).any():
        raise ValueError("Grup peserta berbeda antarwindow.")
    return frames, hashes, paths


def split_frame(frame, hashes, reference):
    """Reuse historical assignments only for exactly matching dataset hashes."""
    result = frame.copy()
    if hashes == reference["dataset_sha256"]:
        records = reference["development"]["window_ids"]["25"]
        mapping = {r["window_id"]: r["split"] for r in records}
        if set(mapping) != set(result.window_id):
            raise ValueError("Snapshot split tidak cocok dengan window_id.")
        result["_split"] = result.window_id.map(mapping)
        protocol = "batch05_train_only_historical_reference"
    else:
        # A new dataset gets a new, fixed, participant-separated development
        # assignment. Test is reserved and is never evaluated by this notebook.
        groups = result.participant_id.to_numpy()
        if len(set(groups)) < 8:
            raise ValueError("Dataset baru memerlukan >=8 orang untuk split grup tiga bagian.")
        outer, test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(result, groups=groups))
        fit_pos, val_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43).split(result.iloc[outer], groups=groups[outer]))
        result["_split"] = "test"
        result.loc[outer[fit_pos], "_split"] = "train"
        result.loc[outer[val_pos], "_split"] = "validation"
        protocol = "new_dataset_fixed_participant_split_42_43"
        a = set(result.loc[result._split.eq("train"), "participant_id"])
        b = set(result.loc[result._split.eq("validation"), "participant_id"])
        c = set(result.loc[result._split.eq("test"), "participant_id"])
        if a & b or a & c or b & c:
            raise ValueError("Grup orang bocor antarbagian.")
    if set(result.loc[result._split.eq("train"), "_y"]) != {0, 1, 2}:
        raise ValueError("Train tidak mempunyai tiga kelas; split tidak dicari ulang.")
    return result, protocol


def audit_baselines(frame, source_evidence):
    """Choose references using time/quality metadata, never prediction scores.

    Existing source evidence contains first-task boundaries from canonical event
    ledgers and baseline completion from hrv.csv. The Drive CSV remains the
    authority for feature values. Unknown sources are reported, not guessed.
    """
    records, references = [], {}
    for sid, group in frame.groupby("session_id", sort=True):
        record = {"session_id": str(sid), "participant_id": str(group.participant_id.iloc[0])}
        source = source_evidence.get(str(sid))
        hrv_reasons, eye_reasons = [], []
        reference = {}
        if source is None:
            hrv_reasons.append("source_timing_not_audited")
            eye_reasons.append("source_timing_not_audited")
        else:
            if str(source["participant_id"]) != str(group.participant_id.iloc[0]):
                raise ValueError(f"Grup orang berbeda dari audit sumber: {sid}")
            first_task_ns = int(source["first_task_ns"])
            end_ns = int(source["baseline_end_ns"])
            record.update({
                "baseline_end_ns": end_ns, "first_task_ns": first_task_ns,
                "baseline_duration_seconds": source["baseline_duration_seconds"],
                "source_evidence": "embedded_canonical_source_audit",
            })
            usable = bool_values(group.baseline_usable)
            base = pd.to_numeric(group.baseline_rmssd_ms, errors="coerce")
            coverage = pd.to_numeric(group.baseline_coverage, errors="coerce")
            if not usable.all() or not np.isfinite(base).all() or not base.gt(0).all():
                hrv_reasons.append("baseline_unusable_in_csv")
            if not coverage.ge(0.8 - 1e-9).all():
                hrv_reasons.append("baseline_coverage_below_80_percent")
            if source["baseline_status"] != "ready":
                hrv_reasons.append("source_baseline_not_ready")
            if end_ns > first_task_ns:
                hrv_reasons.append("baseline_overlaps_first_task")
            if usable.all() and (
                not np.allclose(base, source["baseline_rmssd_ms"], rtol=1e-9, atol=1e-8)
                or not np.allclose(coverage, source["baseline_coverage"], rtol=1e-9, atol=1e-8)
            ):
                hrv_reasons.append("csv_baseline_differs_from_source_audit")
            if not hrv_reasons:
                reference.update({"rmssd": float(base.iloc[0]), "hrv_available_ns": end_ns})
                record["baseline_rmssd_ms"] = float(base.iloc[0])
            # A 25-second sensor window is a pre-task reference, not a verified
            # resting measurement. It must precede even practice/first set.
            candidates = group[
                group.study_phase.eq("pre_task")
                & group._end.lt(first_task_ns)
                & group._start.ge(end_ns)
            ].sort_values("_end")
            if candidates.empty:
                eye_reasons.append("no_eligible_initial_25s_pretask_window")
            else:
                row = candidates.iloc[0]
                if row[["ear_left_mean", "ear_right_mean", "iris_size_estimate_mean"]].le(0).any():
                    eye_reasons.append("nonpositive_eye_reference")
                else:
                    reference.update({
                        "eye_values": {col: float(row[col]) for col in CORE},
                        "eye_window_id": str(row.window_id),
                        "eye_available_ns": int(row._end),
                    })
                    record.update({"eye_window_id": str(row.window_id), "eye_reference_end_ns": int(row._end)})
            if hrv_reasons:
                eye_reasons.append("hrv_reference_unusable")
        record.update({
            "hrv_eligible": not hrv_reasons,
            "hrv_rejection_reason": ";".join(hrv_reasons),
            "joint_eligible": not eye_reasons,
            "joint_rejection_reason": ";".join(eye_reasons),
        })
        if not hrv_reasons:
            references[str(sid)] = reference
        records.append(record)
    return pd.DataFrame(records), references


def make_cohort(frame, audit, references, cohort):
    column = "hrv_eligible" if cohort == "hrv_rmssd" else "joint_eligible"
    eligible = set(audit.loc[audit[column], "session_id"])
    result = frame[frame.session_id.isin(eligible)].copy()
    kept = []
    for index, row in result.iterrows():
        ref = references[row.session_id]
        available = ref["hrv_available_ns"]
        if cohort == "hrv_eye":
            available = max(available, ref["eye_available_ns"])
        # Separate the complete feature support from the calibration interval.
        # The 1-second margin also avoids endpoint-rounding overlap.
        if int(row._start) >= available + 10**9:
            kept.append(index)
    return result.loc[kept].reset_index(drop=True)


def representation(frame, references, cohort, mode):
    x = frame[CORE].to_numpy(float).copy()
    if mode == "raw":
        return x
    if mode != "baseline_relative":
        raise ValueError("Representasi tidak dikenal.")
    for pos, row in enumerate(frame.itertuples(index=False)):
        ref = references[str(row.session_id)]
        # Only recorded RMSSD has an independent baseline in the supplied CSVs.
        x[pos, 0] = x[pos, 0] / ref["rmssd"] - 1.0
        if cohort == "hrv_eye":
            b = ref["eye_values"]
            x[pos, 2] -= b["blink_rate_per_minute"]  # zero blink baseline is valid
            for mean, std in ((3, 4), (5, 6), (7, 8)):
                denominator = b[CORE[mean]]
                if denominator <= 0:
                    raise ValueError("Baseline mean harus positif.")
                x[pos, mean] = x[pos, mean] / denominator - 1.0
                x[pos, std] /= denominator
            # Recorded delta is dimensionless; centering preserves its units.
            x[pos, 9] -= b["iris_size_delta_mean"]
    if not np.isfinite(x).all():
        raise ValueError("Normalisasi menghasilkan missing/infinity.")
    return x


def metrics(y, prediction):
    return {
        "rows": int(len(y)),
        "accuracy": float(accuracy_score(y, prediction)),
        "macro_f1": float(f1_score(y, prediction, labels=[0, 1, 2], average="macro", zero_division=0)),
        "recall": recall_score(y, prediction, labels=[0, 1, 2], average=None, zero_division=0).tolist(),
        "confusion_matrix": confusion_matrix(y, prediction, labels=[0, 1, 2]).tolist(),
    }


def estimator(family):
    if family == "logistic_regression":
        return LogisticRegression(**MODELS[family])
    if family == "extra_trees":
        return ExtraTreesClassifier(**MODELS[family])
    if family == "dummy":
        return DummyClassifier(**MODELS[family])
    raise ValueError(f"Keluarga model tidak dikenal: {family}")


def fit_predict(x, y, fit_ids, held_ids, family):
    if set(y[fit_ids]) != {0, 1, 2}:
        raise ValueError("Fitting fold tidak mempunyai tiga kelas.")
    scaler = RobustScaler().fit(x[fit_ids])
    model = estimator(family)
    with threadpool_limits(limits=2):
        model.fit(scaler.transform(x[fit_ids]), y[fit_ids])
        prediction = model.predict(scaler.transform(x[held_ids]))
        fitting = model.predict(scaler.transform(x[fit_ids]))
    return prediction, fitting, {"scaler": scaler, "estimator": model}


def make_folds(train, reference, protocol):
    folds = {"participant_held_out": []}
    groups = train.participant_id.to_numpy()
    if len(set(groups)) < 4:
        return {}, "fewer_than_four_train_participants"
    y = train._y.to_numpy()
    for number, (fit_ids, held_ids) in enumerate(LeaveOneGroupOut().split(train, y, groups)):
        if set(y[fit_ids]) != {0, 1, 2}:
            return {}, "participant_fold_training_missing_class"
        if set(groups[fit_ids]) & set(groups[held_ids]):
            raise ValueError("Peserta bocor ke fold fitting.")
        folds["participant_held_out"].append((number, fit_ids, held_ids))
    if protocol.startswith("batch05"):
        historical = []
        for fold in reference["development"]["cv"]["folds"]:
            a, b = set(fold["fit_endpoint_keys"]), set(fold["validation_endpoint_keys"])
            fit_ids = np.flatnonzero(train._key.isin(a))
            held_ids = np.flatnonzero(train._key.isin(b))
            if not len(fit_ids) or not len(held_ids) or set(y[fit_ids]) != {0, 1, 2}:
                historical = []
                break
            # Independently verify the historical time separation after subset selection.
            for sid in set(train.iloc[fit_ids].session_id) & set(train.iloc[held_ids].session_id):
                left = train.iloc[fit_ids].loc[lambda f: f.session_id.eq(sid), "_end"].to_numpy()
                right = train.iloc[held_ids].loc[lambda f: f.session_id.eq(sid), "_end"].to_numpy()
                if np.abs(left[:, None] - right).min() < 45 * 10**9:
                    raise ValueError("Guard fold historis kurang dari 45 detik.")
            historical.append((int(fold["fold"]), fit_ids, held_ids))
        if historical:
            covered = np.concatenate([r[2] for r in historical])
            if sorted(covered.tolist()) != list(range(len(train))):
                raise ValueError("Fold historis tidak mencakup Train tepat sekali.")
            folds["historical_block_cv"] = historical
    return folds, ""


def prediction_table(frame, prediction, cohort, protocol, family, mode, fold):
    result = frame[["window_id", "participant_id", "session_id", "study_phase", "cognitive_load_label"]].copy()
    result["predicted_label"] = [LABELS[int(v)] for v in prediction]
    result["cohort"], result["protocol"] = cohort, protocol
    result["family"], result["representation"], result["fold"] = family, mode, fold
    return result


def evaluate_cohort(frame, references, cohort, reference, protocol, run):
    train = frame[frame._split.eq("train")].reset_index(drop=True)
    folds, reason = make_folds(train, reference, protocol)
    if reason:
        return {"status": "skipped", "reason": reason}, [], [], []
    y = train._y.to_numpy()
    summaries, fold_records, predictions = [], [], []
    # Freeze endpoint assignments and model budget before any fitting.
    manifest = {
        name: [{"fold": n, "fit_window_ids": train.iloc[a].window_id.tolist(), "held_window_ids": train.iloc[b].window_id.tolist()} for n, a, b in cases]
        for name, cases in folds.items()
    }
    save_json(run / f"{cohort}_fold_manifest.json", manifest)
    maximum_fits = sum(len(cases) for cases in folds.values()) * 5
    print(f"{cohort}: Train={len(train)}, orang={train.participant_id.nunique()}, batas CV={maximum_fits} fitting", flush=True)
    count = 0
    for name, cases in folds.items():
        for family in FAMILIES:
            modes = ("raw",) if family == "dummy" else ("raw", "baseline_relative")
            for mode in modes:
                x = representation(train, references, cohort, mode)
                oof = np.full(len(train), -1, dtype=int)
                for number, fit_ids, held_ids in cases:
                    prediction, fitting, _ = fit_predict(x, y, fit_ids, held_ids, family)
                    if (oof[held_ids] != -1).any():
                        raise ValueError("Prediksi OOF ditulis dua kali.")
                    oof[held_ids] = prediction
                    scored = metrics(y[held_ids], prediction)
                    fit_scored = metrics(y[fit_ids], fitting)
                    record = {
                        "cohort": cohort, "protocol": name, "family": family,
                        "representation": mode, "fold": number,
                        "held_participants": ";".join(sorted(set(train.iloc[held_ids].participant_id))),
                        "rows": len(held_ids), "fit_rows": len(fit_ids),
                        "accuracy": scored["accuracy"], "macro_f1": scored["macro_f1"],
                        "train_accuracy": fit_scored["accuracy"], "train_macro_f1": fit_scored["macro_f1"],
                    }
                    fold_records.append(record)
                    table = prediction_table(train.iloc[held_ids], prediction, cohort, name, family, mode, number)
                    predictions.append(table)
                    job = run / "jobs" / f"{cohort}_{name}_{family}_{mode}_{number}.json"
                    save_json(job, {**record, "metrics": scored})
                    count += 1
                    if count % 5 == 0:
                        print(f"  {cohort}: {count}/{maximum_fits} fitting CV selesai", flush=True)
                if (oof < 0).any():
                    raise ValueError("OOF tidak lengkap.")
                scored = metrics(y, oof)
                current_folds = [r for r in fold_records if r["protocol"] == name and r["family"] == family and r["representation"] == mode]
                summaries.append({
                    "cohort": cohort, "protocol": name, "family": family, "representation": mode,
                    "rows": len(train), "participants": int(train.participant_id.nunique()),
                    "accuracy": scored["accuracy"], "macro_f1": scored["macro_f1"],
                    "recall_low": scored["recall"][0], "recall_medium": scored["recall"][1], "recall_high": scored["recall"][2],
                    "mean_participant_accuracy": float(np.mean([r["accuracy"] for r in current_folds])),
                    "fold_macro_f1_std": float(np.std([r["macro_f1"] for r in current_folds], ddof=1)),
                })
    return {"status": "complete", "cv_fits": count, "train_rows": len(train), "train_participants": int(train.participant_id.nunique())}, summaries, fold_records, predictions


def paired_results(leaderboard, folds):
    records = []
    for keys, group in leaderboard[leaderboard.family.ne("dummy")].groupby(["cohort", "protocol", "family"]):
        a = group[group.representation.eq("raw")].iloc[0]
        b = group[group.representation.eq("baseline_relative")].iloc[0]
        matched = folds[(folds.cohort.eq(keys[0])) & (folds.protocol.eq(keys[1])) & (folds.family.eq(keys[2]))]
        pivot = matched.pivot(index="fold", columns="representation", values="accuracy")
        delta = pivot.baseline_relative - pivot.raw
        records.append({
            "cohort": keys[0], "protocol": keys[1], "family": keys[2],
            "raw_accuracy": float(a.accuracy), "relative_accuracy": float(b.accuracy),
            "accuracy_delta_pp": float(100 * (b.accuracy - a.accuracy)),
            "raw_macro_f1": float(a.macro_f1), "relative_macro_f1": float(b.macro_f1),
            "macro_f1_delta_pp": float(100 * (b.macro_f1 - a.macro_f1)),
            "folds_improved_accuracy": int((delta > 1e-12).sum()),
            "folds_worse_accuracy": int((delta < -1e-12).sum()), "folds": int(len(delta)),
        })
    return pd.DataFrame(records)


def lock_candidates(leaderboard):
    locks = {}
    main = leaderboard[(leaderboard.protocol.eq("participant_held_out")) & leaderboard.family.ne("dummy")]
    for cohort, group in main.groupby("cohort"):
        locks[cohort] = {}
        for mode, rows in group.groupby("representation"):
            rows = rows[(rows[["recall_low", "recall_medium", "recall_high"]] > 0).all(axis=1)]
            if rows.empty:
                continue
            selected = rows.sort_values(["macro_f1", "accuracy", "family"], ascending=[False, False, True]).iloc[0]
            locks[cohort][mode] = {"family": str(selected.family), "oof_macro_f1": float(selected.macro_f1)}
    return locks


def evaluate_validation(cohorts, references, locks, run):
    records, tables = [], []
    for cohort, candidates in locks.items():
        frame = cohorts[cohort].reset_index(drop=True)
        fit_ids = np.flatnonzero(frame._split.eq("train"))
        held_ids = np.flatnonzero(frame._split.eq("validation"))
        if not len(held_ids):
            continue
        for mode, chosen in candidates.items():
            x = representation(frame, references, cohort, mode)
            y = frame._y.to_numpy()
            predicted, fitting, bundle = fit_predict(x, y, fit_ids, held_ids, chosen["family"])
            # Serialize only the selected model. No Test features are transformed.
            bundle.update({
                "version": VERSION, "cohort": cohort, "representation": mode,
                "features": CORE, "labels": LABELS,
                "baseline_contract": "New sessions supply audited initial reference; no session-ID lookup learned by the classifier.",
            })
            path = run / "models" / f"{cohort}_{mode}.joblib"
            path.parent.mkdir(exist_ok=True)
            joblib.dump(bundle, path)
            scored = metrics(y[held_ids], predicted)
            records.append({
                "cohort": cohort, "representation": mode, "family": chosen["family"],
                "rows": len(held_ids), "accuracy": scored["accuracy"], "macro_f1": scored["macro_f1"],
                "recall_low": scored["recall"][0], "recall_medium": scored["recall"][1], "recall_high": scored["recall"][2],
                "train_accuracy": metrics(y[fit_ids], fitting)["accuracy"],
                "model_sha256": sha256_file(path),
            })
            tables.append(prediction_table(frame.iloc[held_ids], predicted, cohort, "validation", chosen["family"], mode, -1))
    return pd.DataFrame(records), tables


def export_figures(run, leaderboard, paired):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    directory = run / "figures"
    directory.mkdir(exist_ok=True)
    view = leaderboard[leaderboard.protocol.eq("participant_held_out")].copy()
    view["name"] = view.cohort + " / " + view.family + " / " + view.representation
    fig, ax = plt.subplots(figsize=(11, max(4, len(view) * 0.42)))
    ax.barh(view.name, view.macro_f1 * 100, color=np.where(view.representation.eq("raw"), "#68859b", "#159b88"))
    ax.set(xlabel="Macro-F1 OOF (%)", title="Batch 06: participant held out, Train only", xlim=(0, 100))
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(directory / "participant_oof.png", dpi=170)
    plt.close(fig)
    view = paired[paired.protocol.eq("participant_held_out")]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(view.cohort + " / " + view.family, view.macro_f1_delta_pp, color="#159b88")
    ax.axvline(0, color="black", linewidth=1)
    ax.set(xlabel="Macro-F1 relative minus raw (percentage points)", title="Paired baseline comparison on identical windows")
    fig.tight_layout()
    fig.savefig(directory / "paired_change.png", dpi=170)
    plt.close(fig)


def write_report(run, audit, cohort_status, leaderboard, paired, validation, protocol):
    lines = [
        "# Hasil eksperimen baseline Batch 06", "",
        "Window 25 detik, 10 fitur core, model umum Logistic Regression dan Extra Trees. "
        "Pembanding raw dan baseline-relative memakai baris serta fold identik. "
        "Test tidak dievaluasi. Hasil ini merupakan evaluasi development eksploratif.", "",
        f"Protokol split: `{protocol}`.", "",
        "## Ketersediaan acuan awal", "",
        f"Sesi dalam CSV: {len(audit)}; baseline RMSSD layak: {int(audit.hrv_eligible.sum())}; "
        f"baseline RMSSD dan acuan eye awal layak: {int(audit.joint_eligible.sum())}.", "",
        "`hrv_rmssd` mengganti RMSSD dengan RMSSD / baseline - 1; sembilan fitur lain tetap. "
        "`hrv_eye` juga menormalkan statistik eye terhadap window pre_task murni 25 detik. "
        "SDNN tetap mentah karena CSV tidak menyimpan baseline SDNN independen. "
        "Pre_task adalah acuan sensor sebelum tugas, bukan bukti kondisi istirahat fisiologis.", "",
        "Batas awal tugas dan akhir baseline berasal dari audit sumber kanonis yang disertakan. "
        "Nilai baseline RMSSD dan coverage harus cocok dengan CSV Drive. Sumber baru yang belum "
        "diaudit dikeluarkan dan dilaporkan. Notebook tidak mengklaim telah membaca ulang sumber mentah di Drive.", "",
        "## Perbandingan peserta yang ditahan dari fitting", "",
        "| Subset | Model | Representasi | Accuracy OOF | Macro-F1 OOF | Recall Low/Medium/High |", "|---|---|---|---:|---:|---|",
    ]
    for row in leaderboard[leaderboard.protocol.eq("participant_held_out")].itertuples():
        recalls = "/".join(f"{getattr(row, 'recall_' + label.lower()):.2%}" for label in LABELS)
        lines.append(f"| {row.cohort} | {row.family} | {row.representation} | {row.accuracy:.2%} | {row.macro_f1:.2%} | {recalls} |")
    lines.extend(["", "## Perubahan berpasangan", "", "| Subset | Model | Delta accuracy | Delta Macro-F1 | Fold accuracy naik/total |", "|---|---|---:|---:|---:|"])
    for row in paired[paired.protocol.eq("participant_held_out")].itertuples():
        lines.append(f"| {row.cohort} | {row.family} | {row.accuracy_delta_pp:+.2f} pp | {row.macro_f1_delta_pp:+.2f} pp | {row.folds_improved_accuracy}/{row.folds} |")
    lines.extend(["", "OOF menggabungkan prediksi held-out setiap window Train tepat sekali. "
                  "Fold seorang peserta dapat tidak mempunyai ketiga kelas; Macro-F1 per fold memakai tiga kelas tetap. "
                  "Window overlap bukan sampel independen. Selisih deskriptif bukan bukti signifikansi statistik.", "",
                  "## Validation kandidat terkunci", "",
                  "Model dipilih menurut OOF peserta sebelum validation. Tidak ada pemilihan ulang menurut validation.", "",
                  "| Subset | Representasi | Model | Baris | Accuracy | Macro-F1 |", "|---|---|---|---:|---:|---:|"])
    for row in validation.itertuples():
        lines.append(f"| {row.cohort} | {row.representation} | {row.family} | {row.rows} | {row.accuracy:.2%} | {row.macro_f1:.2%} |")
    lines.extend(["", "## Interpretasi", "",
                  "Manfaat baseline dinilai dari perbandingan raw-relative pada subset dan protokol yang sama. "
                  "Skor subset kecil tidak dibandingkan langsung dengan seluruh data batch sebelumnya. "
                  "Subset eye dengan hanya beberapa orang merupakan kontrol eksploratif dengan ketidakpastian besar. "
                  "Validation historis masih berisi sesi yang dikenal, sedangkan OOF peserta menahan seluruh rekaman orang tersebut. "
                  "Dataset baru memakai split berbasis orang untuk Train, validation, dan Test.", "",
                  "Baseline RMSSD recorder umumnya lebih panjang daripada window tugas 25 detik; "
                  "rasio ini menguji representasi relatif, bukan kesetaraan pengukuran HRV dua durasi. "
                  "Belum ada rating beban mental independen. Perbaikan skor tidak memvalidasi label difficulty secara fisiologis.", "",
                  "## Status subset", "", "```json", json.dumps(cohort_status, indent=2), "```", ""])
    (run / "RESULTS.md").write_text("\n".join(lines), encoding="utf-8")


def run_experiment(dataset_root, output_root, reference, source_evidence, implementation_sha256):
    frames, hashes, paths = load_views(dataset_root)
    frame, protocol = split_frame(frames[25], hashes, reference)
    # Test rows are retained only in the read-only split ledger. All baseline
    # extraction and fitting below operate on Train/validation or pre-task refs.
    audit, references = audit_baselines(frame, source_evidence)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run = Path(output_root) / f"baseline_{stamp}_{uuid.uuid4().hex[:8]}"
    run.mkdir(parents=True, exist_ok=False)
    audit.to_csv(run / "baseline_audit.csv", index=False)
    save_json(run / "baseline_references.json", references)
    frame[["window_id", "participant_id", "session_id", "_split"]].to_csv(run / "split_assignments.csv", index=False)
    plan = {
        "version": VERSION, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_sha256": hashes, "implementation_sha256": implementation_sha256,
        "reference_sha256": object_hash(reference), "source_evidence_sha256": object_hash(source_evidence),
        "split_protocol": protocol, "window_seconds": 25, "features": CORE,
        "cohorts": ["hrv_rmssd", "hrv_eye"], "models": MODELS,
        "selection": "participant_held_out_oof_macro_f1_then_accuracy_positive_recall_all_classes",
        "test_evaluation": False, "search": False, "source_root_required": False,
        "minimum_train_participants": 4, "reference_margin_seconds": 1,
        "versions": {name: importlib.metadata.version(name) for name in ("numpy", "pandas", "scikit-learn", "joblib", "matplotlib")},
        "python": platform.python_version(),
    }
    save_json(run / "plan.json", plan)
    save_json(run / "status.json", {"stage": "fitting", "complete": False})
    print("Run:", run, flush=True)
    print(audit[["participant_id", "hrv_eligible", "joint_eligible", "hrv_rejection_reason", "joint_rejection_reason"]].to_string(index=False), flush=True)
    all_summaries, all_folds, all_predictions = [], [], []
    statuses, cohorts = {}, {}
    for cohort in ("hrv_rmssd", "hrv_eye"):
        # Never transform or fit Test, including when constructing representations.
        pool = frame[frame._split.isin(("train", "validation"))].copy()
        cohort_frame = make_cohort(pool, audit, references, cohort)
        cohorts[cohort] = cohort_frame
        status, summaries, folds, predictions = evaluate_cohort(cohort_frame, references, cohort, reference, protocol, run)
        statuses[cohort] = status
        all_summaries.extend(summaries)
        all_folds.extend(folds)
        all_predictions.extend(predictions)
    if not all_summaries:
        save_json(run / "status.json", {"stage": "baseline_unavailable", "complete": False, "cohorts": statuses})
        raise ValueError(f"Tidak ada subset layak. Baca {run / 'baseline_audit.csv'}; jangan membuat baseline dari tugas.")
    leaderboard, fold_table = pd.DataFrame(all_summaries), pd.DataFrame(all_folds)
    leaderboard.to_csv(run / "cv_leaderboard.csv", index=False)
    fold_table.to_csv(run / "fold_metrics.csv", index=False)
    pd.concat(all_predictions, ignore_index=True).to_csv(run / "oof_predictions.csv", index=False)
    paired = paired_results(leaderboard, fold_table)
    paired.to_csv(run / "paired_comparison.csv", index=False)
    locks = lock_candidates(leaderboard)
    save_json(run / "model_lock.json", {"selected": locks, "plan_sha256": object_hash(plan), "validation_evaluated": False, "test_evaluated": False})
    validation, validation_predictions = evaluate_validation(cohorts, references, locks, run)
    validation.to_csv(run / "validation_metrics.csv", index=False)
    if validation_predictions:
        pd.concat(validation_predictions, ignore_index=True).to_csv(run / "validation_predictions.csv", index=False)
    for key, path in paths.items():
        if sha256_file(path) != hashes[key]:
            raise ValueError("CSV sumber berubah selama run.")
    export_figures(run, leaderboard, paired)
    write_report(run, audit, statuses, leaderboard, paired, validation, protocol)
    save_json(run / "status.json", {
        "stage": "complete", "complete": True, "cohorts": statuses,
        "validation_fits": len(validation), "test_evaluated": False,
        "dataset_unchanged": True, "plan_sha256": object_hash(plan),
    })
    print("\nPerbandingan utama:", flush=True)
    print(paired[paired.protocol.eq("participant_held_out")].to_string(index=False), flush=True)
    print("\nValidation:", flush=True)
    print(validation.to_string(index=False), flush=True)
    print("\nLaporan:", run / "RESULTS.md", flush=True)
    return {"run_dir": run, "leaderboard": leaderboard, "paired": paired, "validation": validation, "baseline_audit": audit}

## Empat representasi tetap

Raw mempertahankan semua fitur. RMSSD-only mengganti RMSSD dengan
RMSSD/baseline − 1. Eye-only menormalkan statistik eye terhadap window pre_task
awal, sementara RMSSD dan SDNN tetap mentah. Gabungan mengaktifkan keduanya.
Tidak ada fitur yang dihapus, baseline nol blink tidak menjadi pembagi, dan
iris tetap geometri iris, bukan pengukuran pupil.

Metrik menyertakan support kelas dan bobot sama per orang. Recall kelas tanpa
contoh ditampilkan N/A; Macro-F1 tetap memakai tiga label untuk konsistensi.

In [ ]:
"""Exp02 definitions, embedded after the unchanged Exp01 helper definitions.

Four fixed representations isolate the transforms on one frozen cohort. This
module uses the shared functions embedded in the preceding notebook cell; it is
not an additional Colab runtime dependency.
"""

import warnings


ABLATION_VERSION = "batch06-exp02-ablation-v1"
ABLATION_MODES = ("raw", "rmssd_only", "eye_only", "combined")
ABLATION_CONTRASTS = (
    ("rmssd_only", "raw"), ("eye_only", "raw"), ("combined", "raw"),
    ("combined", "rmssd_only"), ("combined", "eye_only"),
)


def ablation_representation(frame, references, mode):
    if mode == "raw":
        return representation(frame, references, "hrv_eye", "raw")
    if mode == "rmssd_only":
        return representation(frame, references, "hrv_rmssd", "baseline_relative")
    if mode in ("eye_only", "combined"):
        x = representation(frame, references, "hrv_eye", "baseline_relative")
        if mode == "eye_only":
            x[:, 0] = frame.rmssd_ms.to_numpy(float)
        return x
    raise ValueError(f"Varian ablasi tidak dikenal: {mode}")


def ablation_metrics(frame, prediction):
    scored = metrics(frame._y.to_numpy(), np.asarray(prediction, dtype=int))
    support = np.bincount(frame._y.to_numpy(), minlength=3)
    scored["three_class_evaluable"] = bool((support > 0).all())
    for i, label in enumerate(LABELS):
        scored["support_" + label.lower()] = int(support[i])
        # The fixed-label Macro-F1 above remains comparable to Exp01. Recall
        # itself is unavailable, rather than zero, when there is no support.
        scored["recall_" + label.lower()] = scored["recall"][i] if support[i] else None
    per_person = []
    for person in sorted(frame.participant_id.unique()):
        mask = frame.participant_id.eq(person).to_numpy()
        per_person.append(metrics(frame.loc[mask, "_y"].to_numpy(), np.asarray(prediction)[mask]))
    scored["participants"] = int(len(per_person))
    scored["mean_participant_accuracy"] = float(np.mean([r["accuracy"] for r in per_person]))
    scored["mean_participant_macro_f1"] = float(np.mean([r["macro_f1"] for r in per_person]))
    return scored


def ablation_metric_row(frame, prediction, protocol, mode, family="extra_trees"):
    scored = ablation_metrics(frame, prediction)
    return {
        "protocol": protocol, "representation": mode, "family": family,
        **{key: value for key, value in scored.items() if key not in ("recall", "confusion_matrix")},
    }


def prepare_ablation(dataset_root, snapshot):
    views, hashes, paths = load_views(dataset_root)
    if hashes != snapshot["dataset_sha256"]:
        raise ValueError(
            "Hash CSV Drive berbeda dari run Exp01. Exp02 memerlukan data yang sama "
            "untuk kontrol baris/fold identik; siapkan manifest eksperimen baru "
            "jika dataset memang diperbarui, jangan mengganti split otomatis."
        )
    frame = views[25]
    train_ids, validation_ids = snapshot["train_window_ids"], snapshot["validation_window_ids"]
    if len(set(train_ids)) != len(train_ids) or len(set(validation_ids)) != len(validation_ids):
        raise ValueError("ID snapshot berulang.")
    if set(train_ids) & set(validation_ids):
        raise ValueError("Train dan validation snapshot bertabrakan.")
    reserved = set(snapshot["test_window_ids"])
    if reserved & (set(train_ids) | set(validation_ids)):
        raise ValueError("ID Test masuk pembanding.")
    indexed = frame.set_index("window_id", drop=False)
    train = indexed.loc[train_ids].reset_index(drop=True)
    validation = indexed.loc[validation_ids].reset_index(drop=True)
    refs = snapshot["baseline_references"]
    evidence = snapshot["source_evidence"]
    if set(train.participant_id) != set(snapshot["participants"]):
        raise ValueError("Peserta Train berubah dari snapshot.")
    if set(train._y) != {0, 1, 2}:
        raise ValueError("Train tidak memuat tiga kelas.")
    # Audit the same references against the values in the Drive CSV. No reference
    # can be selected anew after viewing ablation scores.
    for sid, ref in refs.items():
        source = evidence[sid]
        if int(source["baseline_end_ns"]) > int(source["first_task_ns"]):
            raise ValueError("Baseline RMSSD bertabrakan dengan tugas.")
        rows = frame[frame.session_id.eq(sid)]
        if not bool_values(rows.baseline_usable).all() or not rows.baseline_coverage.ge(0.8 - 1e-9).all():
            raise ValueError("Baseline RMSSD tidak layak pada CSV.")
        if not np.allclose(rows.baseline_rmssd_ms, ref["rmssd"], rtol=1e-9, atol=1e-8):
            raise ValueError("Baseline RMSSD berubah.")
        initial = indexed.loc[ref["eye_window_id"]]
        if initial.study_phase != "pre_task" or int(initial._end) >= int(source["first_task_ns"]):
            raise ValueError("Acuan eye bukan window sebelum tugas pertama.")
        if int(initial._start) < int(source["baseline_end_ns"]):
            raise ValueError("Acuan eye mendahului penyelesaian baseline recorder.")
        if int(initial._end) != int(ref["eye_available_ns"]):
            raise ValueError("Waktu akhir acuan eye berubah.")
        for col in CORE:
            if not np.isclose(initial[col], ref["eye_values"][col], rtol=1e-9, atol=1e-8):
                raise ValueError(f"Nilai acuan eye berubah: {col}")
        for part in (train, validation):
            subset = part[part.session_id.eq(sid)]
            if not subset.participant_id.eq(source["participant_id"]).all():
                raise ValueError("Grup peserta berbeda dari bukti sumber.")
            available = max(int(ref["hrv_available_ns"]), int(ref["eye_available_ns"]))
            if not subset._start.ge(available + 10**9).all():
                raise ValueError("Window supervised bertabrakan dengan acuan awal.")
    if set(pd.concat([train, validation]).session_id) != set(refs):
        raise ValueError("Sumber pembanding tidak cocok dengan baseline.")
    position = dict(zip(train.window_id, range(len(train))))
    folds = {}
    for protocol, cases in snapshot["folds"].items():
        folds[protocol] = []
        covered = []
        for case in cases:
            if len(set(case["fit_window_ids"])) != len(case["fit_window_ids"]) or len(set(case["held_window_ids"])) != len(case["held_window_ids"]):
                raise ValueError("ID fold berulang.")
            fit = np.array([position[key] for key in case["fit_window_ids"]], dtype=int)
            held = np.array([position[key] for key in case["held_window_ids"]], dtype=int)
            if set(fit) & set(held) or not len(held) or set(train.iloc[fit]._y) != {0, 1, 2}:
                raise ValueError("Fitting/held-out fold tidak sah.")
            if protocol == "participant_held_out":
                if set(train.iloc[fit].participant_id) & set(train.iloc[held].participant_id):
                    raise ValueError("Peserta bocor ke fitting.")
                if train.iloc[held].participant_id.nunique() != 1:
                    raise ValueError("Fold utama harus menahan tepat satu orang.")
            elif protocol == "historical_block_cv":
                for sid in set(train.iloc[fit].session_id) & set(train.iloc[held].session_id):
                    a = train.iloc[fit].loc[lambda f: f.session_id.eq(sid), "_end"].to_numpy()
                    b = train.iloc[held].loc[lambda f: f.session_id.eq(sid), "_end"].to_numpy()
                    if np.abs(a[:, None] - b).min() < 45 * 10**9:
                        raise ValueError("Guard waktu kurang dari 45 detik.")
            else:
                raise ValueError("Protokol snapshot tidak dikenal.")
            covered.extend(held.tolist())
            folds[protocol].append((int(case["fold"]), fit, held))
        if sorted(covered) != list(range(len(train))):
            raise ValueError("OOF tidak mencakup Train tepat sekali.")
    if set(folds) != {"participant_held_out", "historical_block_cv"}:
        raise ValueError("Dua protokol tetap harus tersedia.")
    return {"train": train, "validation": validation, "references": refs, "folds": folds, "hashes": hashes, "paths": paths}


def ablation_predictions(frame, prediction, protocol, mode, fold, family="extra_trees"):
    result = frame[["window_id", "participant_id", "session_id", "study_phase", "cognitive_load_label"]].copy()
    result["predicted_label"] = [LABELS[int(value)] for value in prediction]
    result["protocol"], result["representation"], result["fold"] = protocol, mode, fold
    result["family"] = family
    return result


def ablation_contrasts(summary):
    records = []
    for protocol, rows in summary[summary.family.eq("extra_trees")].groupby("protocol"):
        by_mode = rows.set_index("representation")
        for treatment, control in ABLATION_CONTRASTS:
            a, b = by_mode.loc[treatment], by_mode.loc[control]
            result = {"protocol": protocol, "treatment": treatment, "control": control, "rows": int(a.rows)}
            if a.rows != b.rows or a.participants != b.participants:
                raise ValueError("Kontras tidak memakai endpoint yang sama.")
            for metric in ("accuracy", "macro_f1", "mean_participant_accuracy", "mean_participant_macro_f1"):
                result[metric + "_delta_pp"] = float(100 * (a[metric] - b[metric]))
            records.append(result)
    return pd.DataFrame(records)


def ablation_participant_metrics(predictions):
    records = []
    label_map = dict(zip(LABELS, range(3)))
    for keys, group in predictions.groupby(["protocol", "family", "representation", "participant_id"], sort=True):
        frame = group.assign(_y=group.cognitive_load_label.map(label_map)).reset_index(drop=True)
        prediction = group.predicted_label.map(label_map).to_numpy()
        record = dict(zip(("protocol", "family", "representation", "participant_id"), keys))
        record.update(ablation_metric_row(frame, prediction, keys[0], keys[2], keys[1]))
        records.append(record)
    return pd.DataFrame(records)


def ablation_control_check(summary, snapshot):
    records = []
    for old in snapshot["exp01_controls"]:
        current = summary[summary.protocol.eq(old["protocol"]) & summary.representation.eq(old["representation"]) & summary.family.eq("extra_trees")].iloc[0]
        records.append({
            "protocol": old["protocol"], "representation": old["representation"],
            "exp01_accuracy": old["accuracy"], "exp02_accuracy": float(current.accuracy),
            "exp01_macro_f1": old["macro_f1"], "exp02_macro_f1": float(current.macro_f1),
            "same_metrics": bool(np.isclose(current.accuracy, old["accuracy"], rtol=0, atol=1e-12) and np.isclose(current.macro_f1, old["macro_f1"], rtol=0, atol=1e-12)),
        })
    return pd.DataFrame(records)


def ablation_report(run, summary, validation, contrasts, controls, plan):
    def percent(value):
        return "N/A" if pd.isna(value) else f"{value:.2%}"
    lines = [
        "# Hasil ablasi baseline Batch 06 Exp02", "",
        "Empat representasi memakai peserta, window, fold, dan Extra Trees yang sama dengan subset eye Exp01. "
        "Seluruh fitur tetap ada; yang berubah adalah normalisasi RMSSD dan/atau eye. SDNN tetap mentah. "
        "Test tidak dievaluasi; semua hasil merupakan eksplorasi development pada subset kecil.", "",
        "## OOF peserta yang ditahan dari fitting", "",
        "| Varian | Accuracy window | Macro-F1 tiga kelas | Mean accuracy orang | Mean Macro-F1 orang | Support Low/Medium/High |",
        "|---|---:|---:|---:|---:|---|",
    ]
    for row in summary[summary.protocol.eq("participant_held_out")].itertuples():
        lines.append(f"| {row.representation} | {row.accuracy:.2%} | {row.macro_f1:.2%} | {row.mean_participant_accuracy:.2%} | {row.mean_participant_macro_f1:.2%} | {row.support_low}/{row.support_medium}/{row.support_high} |")
    lines.extend(["", "Macro-F1 per orang memakai tiga kelas tetap, termasuk kelas yang tidak hadir pada fold tertentu. "
                  "Rata-rata orang memberi setiap peserta bobot yang sama; accuracy window memberi setiap baris bobot yang sama.", "",
                  "## Kontras yang dibakukan", "",
                  "| Perlakuan dibanding kontrol | Delta accuracy | Delta Macro-F1 | Delta mean accuracy orang |", "|---|---:|---:|---:|"])
    for row in contrasts[contrasts.protocol.eq("participant_held_out")].itertuples():
        lines.append(f"| {row.treatment} vs {row.control} | {row.accuracy_delta_pp:+.2f} pp | {row.macro_f1_delta_pp:+.2f} pp | {row.mean_participant_accuracy_delta_pp:+.2f} pp |")
    lines.extend(["", "`eye_only vs raw` menilai transformasi eye saat RMSSD tetap asli. "
                  "`combined vs rmssd_only` menilai penambahan transformasi eye saat RMSSD sudah relatif. "
                  "Perbedaan keduanya menunjukkan ketergantungan pada representasi lain, bukan efek fisiologis kausal.", "",
                  "## Validation historis", "",
                  "Validation sudah diperiksa pada Exp01 dan tidak merupakan hold-out baru. "
                  "Semua empat varian dibakukan sebelum fitting; tidak ada pemilihan ulang dari validation.", "",
                  "| Varian | Window | Accuracy | Macro-F1 tiga kelas | Recall Low | Recall Medium | Recall High | Support Low/Medium/High |", "|---|---:|---:|---:|---:|---:|---:|---|"])
    for row in validation.itertuples():
        lines.append(f"| {row.representation} | {row.rows} | {row.accuracy:.2%} | {row.macro_f1:.2%} | {percent(row.recall_low)} | {percent(row.recall_medium)} | {percent(row.recall_high)} | {row.support_low}/{row.support_medium}/{row.support_high} |")
    if not validation.three_class_evaluable.all():
        lines.extend(["", "**Validation tidak mencakup ketiga kelas. Recall kelas dengan support nol adalah N/A. "
                      "Macro-F1 tetap dihitung dengan tiga label untuk konsistensi dan memasukkan F1 nol pada kelas tanpa support. "
                      "Accuracy validation tidak boleh dipromosikan sebagai pengujian tiga kelas lengkap atau pengguna baru.**"])
    lines.extend(["", "## Pemeriksaan kontrol Exp01", "",
                  "Raw dan combined dijalankan ulang dalam Exp02 sebagai kontrol internal. "
                  "Kontras selalu dihitung dari hasil Exp02 yang baru, bukan mencampur skor lintas run.", "",
                  "| Protokol | Varian | Metrik sama dengan Exp01 |", "|---|---|---|"])
    for row in controls.itertuples():
        lines.append(f"| {row.protocol} | {row.representation} | {row.same_metrics} |")
    lines.extend(["", "Perbedaan kontrol perlu dibaca bersama versi paket dan numerik runtime. "
                  "Tidak ada seed, parameter, atau fold yang diubah untuk memaksakan kecocokan.", "",
                  "## Batas kesimpulan", "",
                  "Acuan eye adalah pre_task, bukan pengukuran pupil atau baseline istirahat tervalidasi. "
                  "Label tetap difficulty/fase; belum ada rating beban mental independen. Window overlap bukan observasi independen. "
                  "Empat peserta tidak cukup untuk klaim populasi luas; perubahan deskriptif tidak dianggap signifikansi statistik.", "",
                  f"Budget: {plan['cv_fit_budget']} fitting CV dan {plan['validation_fit_budget']} fitting Train penuh. "
                  "Model validation disimpan sebagai empat pembanding, bukan pemenang final menurut validation.", ""])
    (run / "RESULTS.md").write_text("\n".join(lines), encoding="utf-8")


def ablation_figures(run, summary, contrasts, validation):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    destination = run / "figures"
    destination.mkdir(exist_ok=True)
    data = summary[(summary.protocol.eq("participant_held_out")) & summary.family.eq("extra_trees")].set_index("representation").loc[list(ABLATION_MODES)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = ["#68859b", "#b98d40", "#637ac8", "#159b88"]
    for ax, metric, title in ((axes[0], "macro_f1", "OOF window Macro-F1, three fixed labels"), (axes[1], "mean_participant_accuracy", "OOF mean accuracy, equal participant weights")):
        ax.bar(data.index, data[metric] * 100, color=colors)
        ax.set(ylim=(0, 100), ylabel="Percent", title=title)
        ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()
    fig.savefig(destination / "ablation_oof.png", dpi=170)
    plt.close(fig)
    data = contrasts[contrasts.protocol.eq("participant_held_out")]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(data.treatment + " vs " + data.control, data.macro_f1_delta_pp, color="#159b88")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set(xlabel="Macro-F1 change (percentage points)", title="Paired transforms on identical windows and folds")
    fig.tight_layout()
    fig.savefig(destination / "ablation_contrasts.png", dpi=170)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    row = validation.iloc[0]
    ax.bar(LABELS, [row.support_low, row.support_medium, row.support_high], color=["#68859b", "#c95c58", "#159b88"])
    ax.set(ylabel="Windows", title="Historical validation class support")
    fig.tight_layout()
    fig.savefig(destination / "validation_support.png", dpi=170)
    plt.close(fig)


def run_ablation(dataset_root, output_root, snapshot, implementation_hashes):
    data = prepare_ablation(dataset_root, snapshot)
    train, validation, refs, folds = data["train"], data["validation"], data["references"], data["folds"]
    run = Path(output_root) / f"ablation_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
    run.mkdir(parents=True, exist_ok=False)
    plan = {
        "version": ABLATION_VERSION, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_exp01_run": snapshot["source_run"], "snapshot_sha256": object_hash(snapshot),
        "dataset_sha256": data["hashes"], "implementation_sha256": implementation_hashes,
        "features": CORE, "variants": list(ABLATION_MODES), "model": MODELS["extra_trees"],
        "dummy": MODELS["dummy"], "contrasts": list(ABLATION_CONTRASTS),
        "cv_fit_budget": sum(len(cases) for cases in folds.values()) * 5,
        "validation_fit_budget": 4, "test_evaluated": False,
        "search": False, "validation_model_selection": False,
        "versions": {name: importlib.metadata.version(name) for name in ("numpy", "pandas", "scikit-learn", "joblib", "matplotlib")},
        "python": platform.python_version(),
    }
    save_json(run / "plan.json", plan)
    save_json(run / "cohort_snapshot.json", snapshot)
    save_json(run / "status.json", {"stage": "cv", "complete": False, "test_evaluated": False})
    print("Run:", run, flush=True)
    print("Train:", len(train), "window,", train.participant_id.nunique(), "orang", flush=True)
    val_support = np.bincount(validation._y, minlength=3)
    print("Support validation Low/Medium/High:", val_support.tolist(), flush=True)
    if (val_support == 0).any():
        print("PERHATIAN: validation historis tidak mencakup tiga kelas; recall tanpa support = N/A.", flush=True)
    summaries, fold_rows, tables, fit_count = [], [], [], 0
    for protocol, cases in folds.items():
        for mode in (*ABLATION_MODES, "dummy"):
            family = "dummy" if mode == "dummy" else "extra_trees"
            x = ablation_representation(train, refs, "raw" if mode == "dummy" else mode)
            oof = np.full(len(train), -1, dtype=int)
            for number, fit, held in cases:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    prediction, fitting, _ = fit_predict(x, train._y.to_numpy(), fit, held, family)
                if (oof[held] >= 0).any():
                    raise ValueError("OOF ditulis lebih dari sekali.")
                oof[held] = prediction
                row = ablation_metric_row(train.iloc[held], prediction, protocol, mode, family)
                row.update({"fold": number, "fit_rows": len(fit), "train_accuracy": float(accuracy_score(train.iloc[fit]._y, fitting)), "warnings": [str(w.message) for w in caught]})
                fold_rows.append(row)
                tables.append(ablation_predictions(train.iloc[held], prediction, protocol, mode, number, family))
                save_json(run / "jobs" / f"{protocol}_{mode}_{number}.json", row)
                fit_count += 1
                print(f"CV {fit_count}/{plan['cv_fit_budget']}: {protocol}, {mode}, fold {number}", flush=True)
            if (oof < 0).any():
                raise ValueError("OOF tidak lengkap.")
            summaries.append(ablation_metric_row(train, oof, protocol, mode, family))
    summary = pd.DataFrame(summaries)
    summary.to_csv(run / "cv_leaderboard.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(run / "fold_metrics.csv", index=False)
    predictions = pd.concat(tables, ignore_index=True)
    predictions.to_csv(run / "oof_predictions.csv", index=False)
    # All four fixed variants, including weaker controls, are locked for the
    # historical validation before that endpoint is evaluated.
    save_json(run / "comparison_lock.json", {
        "variants": list(ABLATION_MODES), "plan_sha256": object_hash(plan),
        "oof_sha256": sha256_file(run / "oof_predictions.csv"),
        "validation_evaluated": False, "test_evaluated": False,
        "selection": "all_four_predeclared_controls_no_validation_selection",
    })
    val_rows = []
    joined = pd.concat([train, validation], ignore_index=True)
    fit = np.arange(len(train))
    held = np.arange(len(train), len(joined))
    for mode in ABLATION_MODES:
        x = ablation_representation(joined, refs, mode)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            prediction, fitting, bundle = fit_predict(x, joined._y.to_numpy(), fit, held, "extra_trees")
        bundle.update({"version": ABLATION_VERSION, "representation": mode, "features": CORE, "labels": LABELS,
                       "baseline_contract": "Provide audited initial references for a new session; no task-session statistics are learned as references."})
        path = run / "models" / f"extra_trees_{mode}.joblib"
        path.parent.mkdir(exist_ok=True)
        joblib.dump(bundle, path)
        row = ablation_metric_row(validation, prediction, "validation", mode)
        row.update({"train_accuracy": float(accuracy_score(train._y, fitting)), "model_sha256": sha256_file(path), "warnings": [str(w.message) for w in caught]})
        val_rows.append(row)
        tables.append(ablation_predictions(validation, prediction, "validation", mode, -1))
        print("Validation:", mode, row["accuracy"], flush=True)
    val_table = pd.DataFrame(val_rows)
    val_table.to_csv(run / "validation_metrics.csv", index=False)
    all_predictions = pd.concat(tables, ignore_index=True)
    all_predictions[all_predictions.protocol.eq("validation")].to_csv(run / "validation_predictions.csv", index=False)
    ablation_participant_metrics(all_predictions).to_csv(run / "participant_metrics.csv", index=False)
    contrasts = ablation_contrasts(pd.concat([summary, val_table], ignore_index=True))
    contrasts.to_csv(run / "paired_contrasts.csv", index=False)
    controls = ablation_control_check(pd.concat([summary, val_table], ignore_index=True), snapshot)
    controls.to_csv(run / "exp01_control_checks.csv", index=False)
    if not controls.same_metrics.all():
        print("PERHATIAN: sebagian kontrol berbeda dari Exp01. Baca exp01_control_checks.csv dan versi paket; semua kontras memakai hasil Exp02 saja.", flush=True)
    for key, path in data["paths"].items():
        if sha256_file(path) != data["hashes"][key]:
            raise ValueError("CSV sumber berubah selama run.")
    ablation_figures(run, summary, contrasts, val_table)
    ablation_report(run, summary, val_table, contrasts, controls, plan)
    save_json(run / "status.json", {
        "complete": True, "stage": "complete", "cv_fits": fit_count,
        "validation_fits": len(val_rows), "test_evaluated": False,
        "dataset_unchanged": True, "plan_sha256": object_hash(plan),
        "validation_three_class_evaluable": bool(val_table.three_class_evaluable.all()),
        "exp01_control_metrics_match": bool(controls.same_metrics.all()),
    })
    print("Laporan:", run / "RESULTS.md", flush=True)
    return {"run_dir": run, "leaderboard": summary, "validation": val_table, "contrasts": contrasts, "controls": controls}

ABLATION_IMPLEMENTATION_HASHES = {'shared_exp01': '13c087d6a1a16ccd06117547f1ccf253be174ba5dcea4f34a162dc60b8f00f16', 'exp02': '50fd31a47376c3a7d5f3173b9ed1691bd17609c5f062f5c5c892b5a12e6769ca'}

## Snapshot pembanding

Window, urutan fitting, baseline, serta empat fold peserta dan lima fold waktu
dibekukan dari run Exp01. Seluruh varian memakai daftar yang sama. Validation
lama sudah pernah diperiksa, hanya berisi 43 Low dan 56 High tanpa Medium,
dan bukan hold-out baru. Split ini dipertahankan untuk kontrol yang sama.

In [ ]:
import base64
import zlib

ABLATION_SNAPSHOT_PAYLOAD = (
    'c-ri}-L4(Sv2OV)hHiVCsLY7SA3a51oVyK!phWVRy=BRew72@S+wa~rN&k^+aSn+!&gpJoS}-6`#TDzTh#XlNnGyMyufBQp)0^+!'
    'egEd=k8j?-`SH#7-@f_jtFM3em#_NKM?3m_*{aQ|m;d|GUVi(->v!M(_m@Av`sqL3KlA2)ym|TRzh1rj?$tNny?Ocl?Ju^u<5;_o'
    'sGfbS<2e59ckdtkuUFsw=ih$zH>bh>c=_Y2pWnRv^P3-E{^{NK|M~Nq+h6W4-aq~7$Cux|dHeIrKfQVN{rShoi@uMD*3O@sN7d@_'
    'uTTB-^XvBywWA$*%r#rIKBLy(9{Ta!fB)mZ{;V~7%Oh&$v9q_2p8EUGI`*6+k9Gc}^EE`B554>G-A^xndiQ_5dHMRycR#=S`w#s6'
    '`S6jejd}i+7^`Qj_0i*Re){>{pKe<J`78AEx9q*O*e&|8#`Dkr<Ci(s(bpQg&t5Td&hz;n|Mb&OuV4P@_CkBT*mGANr>k<GKl=R7'
    'KfV6``@j9y`661a++!c3PU-pAzj@@Bci+GM;g^^1ZdS<O|M%^`|Gpi+@AJR4`nn%q$9nO>#sAwezdnvnA8216@zDccomT(&<9~fF'
    'VB({Id9UvF%jb)pSu^_S_L|4K{q7%r)i_?ZJpbvxB{U;*obU1I=imEHq1j*LvyoXX&KQb${%UQ<-p>W*8I!RpM~qWpp5A;uFe_pl'
    'asIsX|7R$#=L2)(8e^S~ZX=GF36a@z=dSY?Tk9MgZ54%Po`Yz`8D{esbBs=*c|O;V=rJ>nbJVPlLo?3zc7|>BJx`DA);<c&I$n%w'
    'N1a39bj$hwd!hMRfBz%$;`nbZzi#vEThzRF@h74Aj|Xyp^uV3ayah~r6fl1e%`^U6uG1gqu<LD{ulD2EthVd;ZLxVq_!&>L&G=2R'
    'IbZZ=V>8b<nKvS~+i!J-%=5u{hDeM!hu%5DcFfNN=NXr~R_uBH$df~#3C?kjg0b`H{rJG4yRq5Mp>u-0#~wY-fTq}tUF`(L8F1qa'
    'nn;Sx^Y>WqG22fcW$uq-v+~6`qa0a%>{dtn7@KE^)cMzsbI^`9R~-eh`K-S`3cZtx{p&a<s`oB_CN|$cQ1#IRcVqLUO6#NlBVhg>'
    'n-wqi2X4;uzwTrIGd5d05udl0_#0v~_tDNDGxPlGzimp7_SxW!w$BOk*!^_sN&aW2^z*5;`Z>3qhL6^skIgeMYqnebp1wLI*R#Rd'
    ';{9mYyT<vymK2>ecN^!dbCUi1?LG&>e+kaf_dd=j$bJrhKyaR43;Q^3kDsINH1u(BwtCT9ozu!udyO@YkE65g^K<9?+h^3BKj?h!'
    '^TGLfhyN}1+V9)EaWVGS=a<9>AAb~_|MI{|?0<dW^TGK(V#Y@i^Y`Fv<HgMj`}D@>Cro1e_4eDpes%s%Z%%*x@l~(!KB!vVRJ@*V'
    'zCFHe-`0Nhy1#lm-hMmAtJizJdNW_Wep@?dd$sG$k!`iN-$Wbz_3QfPoR;6dI(5JP=JreW4?pSKRo{H`?TLjqmGhjJa=pDdxqowd'
    '`y4F!`pug+uk)L8BF=Bl>FU+%bK?Bw+lsZ{zS-Zz?I-2?pH#<q^XB+A-p=FAZE}6{cD{Z6`V?C0_1jmo&Tp;Lsc-Y`o7dl-zu31Y'
    'T5n%4^Y%4)n_|Cu@2~&!&+i@b{m(!C@ZIgb{rg|OdiCwM|M}yqZ~w>pAUVHL$2lXM9z1`ylhfzl`O~X!fBfO)+xFqVpFpg0)?IV0'
    ')4#X>{^uWm`1yx#fB5e9OaA%Qk3YZr_T8UfegE^zKfd|y^~)dr^XHp{AK(1>$2UK{`TpluKfn9o7QJ8n`0AJ2Z}f@zbGAJ{F;0x;'
    'Jk_lI#Qa_rx6g$Wpb_!q=D+>nPv4wNkk`-r4sm`r?L0wu4#G1S*OT8NZ{O!Ta@0PD!uaHO_~W~ue*WRdci+DH?&UY%{qXI7y!`gR'
    'e#;xU{ovC;KM8dEz8g=hpC5L|TJ08u=WjQ^)cTzlFzTa!p8N_Y<<IrdJmY!o5A*WB{st$Z-@X1|=C>L?1NLA4iKoGLL-;XUoecl*'
    '-SAs1zklI>_{S*tuQxybbpDqQlga<_!#SONb9<`2IAQVCd(%Jv`MV!}{@WkNNxu(&{{4Uc>CgY;J?mcNi*fstGu~dm`su?T^%w8Q'
    '>`&jm`;T`&|NgskQuzMwKefFdf^UEL?)3+o&-v>8h<y9*=YLZ5{V4sdYo+|U;_h0hzHaNuTIshNSzx5cRU`H3qV5?ZOE5x8LQv93'
    'NeD_BDM>*|CnYH;>7=9tB|%C`Py*v*2;*eXI2pn?88l9YFir-IlOc?gLE~fy<7Ch{8NxUjG){&vP6myWA&iqj3BLJUg6}sQ$si+w'
    'k|ZTfP*P;%I#woOWeQlCgq0~^WfE4VfR#yDnF3ZOVPy(fnS_;j9V^k{Gd1G+XX?$!0wb5l$@w!5QgV5m+>DTt5R^1ha(SFYl9CjZ'
    'Bq>QnNh2jGDCwjm1tp!7Tp#hYoC{$&7qpxUVL2DHoC{$&mj^`G`M~AqI)6qdBPAHQ{{?w7BD}8qUy#|Ze?dNd<<A%qlq4x>f|4R5'
    'muJSL={JSxH-n-pgy;&2t`wpxD7sRJuAt~jA-aMbpMv8P=9A3p^GOyteh7}IWI+naf}~_Y3dw?`WI+naf~0kP3hVf!b$klz_@}S@'
    '86$#{A|=-cBP9z`NEYNI$1lP0lN`SU$4_$n5*)us$rhAs_~1St2*F4rBbUF0i@t?R_!chu7B1mixac#rgwND!FyhY#x?yC2k;~Cl'
    '4VH5^BbSwQ2;uXA>mf{8q7t%1MG2fr#R&LBNZ=HG5|!{tRP;$y!Y5JD{z(b@Cq>^~C46@keRq}c-BtA6Rl;{y(RWt~-(5xDU3KTX'
    '>o=TnuKQ=4&##rIXPke#kqt&ZIcfa`M$QjJQqs3zgp`D!q>+*klr&P3f|5>3Qc%)KNeN1Vl$4+(?km}!#`qT*$s{9!k|ZTfP*S9%'
    '2}+8TTxZuNc5Q)Oo7lAlc5PzU7TC3kU0Yz+CU$LsU7Ogo1$J#>*B02dfn8^z-ytZu&#s#hK}nL5CMYRV(gY<%N-neOsb8NlvIQfg'
    'Bm^amlw4-l?I6ZoCFcW|*>yXXl4Rtvk_4aJ4r1hWBlmrBGa@Kyq$C9;os?YnNdbjpT&HjpQ21s<P?Dsi2}+8TG(ky`lFJl+>epwC'
    'Y{3XA2|-CCB_Swjq$C9;os^`Yq?3{olmsa$K?$&H3hWxht|_o<5WA+pu0iaY0=ovWYYOa|#I7l@YZAMrz^+N`ngY8fv1<zKqTUxV'
    '1$Iqh*A&<_iCt4**Cci=fnAf>wFGueV%HMbMY|3m>^cPPI)t$65VY$M!mdNmu0sgB4neyPA?!K?rEEh;*#@O-LrB>MrEEh;*#@O-'
    'LrB>MrEEh;*#@O-LrB@?0h8A4{Ph-$Y%o%Sk{~4|D7imr-HyBqO72fuHzR_QBqdEyQlz8_N{W<RX4g}{K4WAHMo38rN*XB%K}jPe'
    'DJbcrBn2g%l$4+(NJ$AwfL&c+S0i?HfnAN*)dhAnVpkW~)rehPU{@!0b%9--*wqDgbz)Z+*wu+$U0_!yc6EVWo!Hd{c6DM`7ueN_'
    'T_Lcm6T3oSS0{Fbz^+d03V~gn*cAf1g4h)TyMovi0=t6P6#~0xt2~9R@}#Zu6t>Er`t|7pB`66}Qi2j-R|@Q+lLaZ9EJ!+8kZ`i#'
    'SGDOqcY^9}o8B{-e>`c^`<so_K}Je2LQ48&CAYsuMoKV3N<vW5NJ$7v8YxLZNhc*KDCwl61SLUAN>CE_m8?%={ELj_K}G~6NlKca'
    'q)15<loTns&aOr5S^~Qkv1<wJTEwm;uxk;!mcXt>>{<f57O`sy>{`UGC9rD|ySBitMeN!FyEcXJ7D9Ma2yY>TH-+#PLU>aMZy|&?'
    'h42<actZ%saXo}@Cw1=|sp~W2?bztu*L6F?Sb~uPBbR5!+p$qna(QOF86hPhC~2f51SO4>q@bjelFL{LTF!;AoC{jcg|M6pTF!;A'
    'oD2Hk4&j44=z}|i5AL83?hrn>gFd)J_}~uu;11z~JLrQugb(hZ5AF~?xPv~pL-^nh`rr=XgFEPhJA@DJpbzd4KDdKExI_5he(Kk!'
    'Mw+0cNJ$fv6e+pRu1)OP0=qV`YYXh6l~xEVt)P`w2rI3il~xEVt)P`w2rI3il~xEVt)P`w8Y`_0?7A5dlq4x>f|4R7O;A##<TAUS'
    '`t=zjb=?T~q+R#P?GV`kMlO$&o00oU`t@;gJ4AMn5kX0k62m7&N}8agNXhka(rKJ@VVuxCu-SzW?i9iygmA$0dpn3xf)P?uf|4L5'
    'B`5)(q~a66N(xvB!b%EQ3BpPWSP8;P3Rp?PN(xv>SUTJe#|TOqDY?9KIJ>;V>nh=O6}_$!URTlUD&=)GQj&s_PD)Zx(n-nZQ2(oM'
    'y8ZyjJ2zc_7)YPqbp6|n6c}m8bt6SenxLdeNx!V*Js`nI2}Vds2ud0$2|-CCB`GNBq$C9;os^WIBuGgKO5(nf`ZUJB$Oy2j1a>8{'
    's|0o>v8x1jC9$gnb``O!1a=j%s|0oxv8x1j6|t)Xb``O!1a=j%s|0oxv8x1j6|rjw>?&f{5ZG13t|73ih+RWqR}s60z^*~;8Unip'
    'v1<tI8pN(4uxk*zhQO{t>>2{Q2C-`h>>9+bA+T!@yQaXdLF}3Wy9Tjq3hWxht|_o<5WA+pu1V~g0=p)$YYOa|#I7l@YZAMrz^+N`'
    'ngY8fv1<zKn#8Utuxk>#mcXt_>{<f5Cb4S??3%=`C9rD}yOzMNMeJGvyB4u)3G7<Lt|hQ*5xbVau0`xx0=pKmYYFUH#I7Z<YZ1G)'
    'z^+B?+5)>4v1<$LTEwm`uxk;!w!p4U?AijmHnD39?ApYxEwF17ySBitP3+nNyEd_F3+&p&t}U=@1H0-F*tLOObqMU*z^*z3c5Pr+'
    '9Rj;Hu&a*i?0V|gXN;6!gp`D!q>+*klr&OunO#MB;S%z~MS0;8^1?-V;S%z~MS0;8^1?-V;S%z~MS0;8^1?-V;S%z~MS0;8^1?-_'
    's1j09MX9I~Qc*>zsG7>_>ZBwEB`_F;WI<7~poC;WQL>;eCkxIk?E9yp&Ic}^ih6J49)&*`DZvOS>6ew<KNa=CNL@DqN=i^tq@)BT'
    '_s>YYe;@=Ujg*9-q>+*olyp*(f|5>3N>CD{qy!~#U&-Y&63_Yd86%gK4D!hkd@{%<L-5I@!8nD%IB76WVK7b_j8hnllN>(<$4_$n'
    'R2&b3aS4NQ(qLS|V4U<xw1iKhNuNYZ_#|57_$4@gk>j`E_(hK2g5wuCehZG@@VeR|ysizes~y7Y+VHyCp}ejlC6~uZJAlIH1D7e>'
    '4j7C#BZ3l$u68}T&Yy9AW^C>H%y=_GN=i@?q@)BT;FGTS1V((nKH{JH^{J62C@E6X1SLgEE-$fLz-MZU>z}DDXfTE_7!$tgTVDUF'
    'Z%MB!h1W&v_*RM$@QJXFZ?ukY!aBauI=%_(_(tpaCamKdt>c@pj&DVP90DMV067Fe76EbyfE<v6Y~y+kvJJ>VwsAcN*#-e}2!I^K'
    't|72%5W9xJF3NW`A>Y*~-_?YCR~y8xDX?o0yQaXdLF}3Wy9Tjq3hWxht|_o<61%3ru1V~g0=p<>+k}*Dqm*qEQnrm!woORcHcHtx'
    'A!XYph49l|>_41V-v032>#x54-EscwAK!fU`sFY0zJLA0FE8J{{^|VFfB#$Wk3Mg`KW5cudw=xzHu{+>STCw)AM2CeAOCnD;-d%t'
    '*<j>17->O9mSBXGq@bjelDw>>lTW(flTJ#yqNJ0Ox~$}EsdisUeEM5_nUMuXLQv93NeD{fzLNa(eSVRV3^F1pNm9}TB}Gb_prlAi'
    'zpNx_oTM;LlEz7Xew_T;3kf@2{oaME-#cCX-i52*J6-+Wg{$8?UH#sLtKU0a{oaME-#cCX-i52*JKf6Og<IJ>-OApDTiH9^%HD-r'
    '**o3J-i2G)JKf6Og<IJ>-OApDTiH9^%HD-r**o3J-i2G)JKf6Og<IJ>-OApDTiH9^%HD-r**o3J-i2G)JKf6Og<IJ>-OApDTiH9^'
    '%HD-r**o3J-i2G)JKf6Og<IJ>-OApDTiH8Z&fSH}xjS9X-G$4!J6+D*h0D2nPza|G!a*UNLI?+ia0($D6v8QlaMJXf!t_hGvUlND'
    '_D;95_Y|hzr0F+>={ITmO=0>?Vpj_6N@7<E>`G!+3G7N@R|)J&Vpj?5N@7<D>`G!+3G6CjR|)JYVpj?5Dq>d&>?&ec3G6CjR|)JY'
    'Vpj?5Dq>d&>?&f{5ZG13t|72%PzVnpga?K25JGrR2oE8I2ZituLU>RJ4<UpHh42(Ycu)vWC4`fdG(ky`k|roAQgVIzoizPUVfvjk'
    '{Z3)}oy4vwuxk>#rogUA?3x0*7CC+ij$h>XB{+VO<JW!1|C;M$Zi%_O*U9v_e#Or%XLgX05{!_N6qIyQl7f;>N=i@?q@)BT_iuCS'
    '!fkGyZgcCxZEl@zbL+xwZk=v(>%whroo;jM!fkGyZgcCxZEl@zbL+xwZk=v(>%whrPyPDzffST<Qj&s_PD)Bp5~QRACBUu{*p<Yt'
    '64;f*t`gXl#I6$9mBg+R*hN<>cHwHpPFE{-;cCTBS1WemYQ;`hD|X>(#ZFf%cHwHpPFE}T64+J5t`gW)#I7N*tB74gU{?{lhQO{O'
    'b`61DMeG^^y9TjqTxVAt5W;a>58*f<gri*#;W!|KZ$>UFX#mJ**8v#~02%E%Afo{wqg@ANG>Wb!L|3Ee>Oyoiimt9iSCWz@C@E6X'
    '1SLgEE(7xX89_>}M^~rl3L&~WMOO&X)hW6{h^|i26+(1%imni%D`5JK5T;*Bs)dkL3rebmkW>pws)dkL3rebmkW>pws)dkL3rebm'
    'kW{<1tG}zH+I64Q0Y0gA-6wT`PpVz_Nj30EHNhthk_FYSCkvkX^%*0Al155WP|`_B3Q9UDx%@3$oqW;-pLFs`7ktvmCtdJKC!ci1'
    'CqYU|P;!5cFX0Qa=nJxhFUX=V$P&IFi@qRB_<}6@f-K<+vgix4gfGaVP5%-${fjpJOW5=;+Vn4B)4yobzn*UT|KYFy_Pf7)_4eJ*'
    'kB%RYjvtSXACHb7zt%`}Xkl?nh8RW`7`fcSBFOO}I6lbnAviwB@nJZgnsG!5N|KbMqNI_M6qIyQl7f;>N-hI3ipEI^<D@9MN{Fta'
    '=&I|{l?O!E`M~AqI)6qdBPAG#AS1%-y5B)6+w~4oPha^nMg%2EN}8ag$jIfHF=_fuVfxLW=n5gas6SZ>{mGL0lcms~EU7<P3jN8F'
    'T8E|3IxMMmSPHGfZWqGeSF(lIwPDh_T~aL=X=LQ`d{Q)@lrWzZ%_k+yCq<|)0rk}&gm2k*!^i?7m!qpFw^BlG<(9#}ucQkhOe?Jt'
    'R$4`ipi0FE_~gza=+QIo(KGJRGw#td?$I;u(KGJRGw#td?$I;u(KGJRGw#td?$I;u(KGJRGw#td?$I;u(KGJRGw#td?$I;u(KGJR'
    'Gw#td?*C^!<9>~im;lIF*8v$502%8#AY%d`V_gU2X{1O=6O<GwxeUmlaT3Bf2^uFMjFX^o62>@L<oGQ(ev#w1;P_3B--6>eIerU{'
    '-{kl$IDW&-cs_7>X1pCyysPAPUTzCUHW(>ENsy8fl-!@RvR$9FZm&Gah@d1%NfVS5DQSX|A|;pEm2@>^3RgoWT@9JS)sRV7L#A*w'
    'WYX1;DO?SibTwoOS3^Ga>(d8HP!go11SP;O;Ve<oS)vrq5+$7_O5rR~(pjPu&Jrb^B}(BeQPNqW6wVSQoh3@)EK$-~q7=>&C7mTo'
    ';Ve<oS)vrq5+$7_O5rR~(pjPu&Jrb^B}(BeQPNqW6wVSQoh3@)EK$-~q7=>&C7mTo;VjWpzdmDR3r0vu2ud0$2|-CCB`GNBq$C9;'
    'os^WIBuGgKN`PI$A+n^iL@AslN;*rF`M68uahJ&BE|JGwB9FU79(Rd6?h<+2CGxmS<Z+kC<1UfMT_TUWL>_mE{Li~f<f&huJ`jSE'
    'MoL0Z(nv`PN;)Y?K}jbiB`66}Qi2j-*A&<_h+R`)*C2LHfn9^xH3fDJV%HSdHHlqQVAmveO@UpL*fj-qO=8y+*foh=Q()I5c1?j@'
    'lh`!{c1>c}64*6~T}xosBz7%<U6a_g1a?hg*Am#Zh+Ruy*CKW;fnAH(wFGu8V%HMbwTN9yVAmpcErDH&*tG<9En?Re*tLjVTVU6w'
    '5Z*!vZwlcpgz%;i-a-g(3gIn;@TL&nLI`iVW$O?^c+)LghZ4d`N}8agNJ$fv6e;P#ouCIPd0e6TxI*=Dh3ex9)yEa8k1JFkSExR&'
    'P<>pX`nW>%o?o9n5P}kLyl@aBXfrc}&CH<9%n&v+gElim*vt&t%nV^OGiWn2gw4#L&CC!sGlMoWL)gp=+6f9_Cn#tqD1@D$pq-!)'
    'c7lR(^&#ZygL3sD<m!WR^&#ZygL3sD<m!WR^&#ZygL3sD<m!WR^&#ZygL3sD<m!WR^&#ZygL3sD<m!WR^&#ZygL3sD<m#XL^{J62'
    'C@E6X1SLgEuCr?sySBitP3+nNyJ(XpgiV^DO_~rkX@WLsLfE7U+N23#lO|}BCWK9zpiP<-Hfe%3X;Rpv3EHGdVUs3klO}~tnxIXZ'
    '6gFv+Hfd7Wq`5WPxv!*M_eoN&K80L;Qm#IQTzyimK80L;Qm#I`;*%mJO;A##<oY=2G)}rOPH00Qg$;qE4S^Ik1d>W}Qz*$zD#=Zu'
    'BsZxfH-(biq%?Y_;uFA%kVa2Rqo<HYPfDYwkVa2Rqo<HYPfDYwkVa2Rqo<HYPb$exp(Hn{3NVE#z@#d`6siD|ssOW;*VRZ#3Q9UD'
    'NkK^`C6`fO^qE@1XX@?h*n^CeV1$%hW>+;BjMc6W#%eGat6d+APyPCg5kW~KB`GNBq$C9;os?W3jGcVa1)p^CNf&(5$tPX#NhhCl'
    '#V0{ZN>FnDTeyVvNzs9&5)LdC9at*iz*5nHr4kM-6&+YA;lNVSfu#}-EEToZDxtMjQEROdT5A=x)+(X3R?#}XKEIA{?Yu@jBaYAY'
    'RBOL)^Y6X#ju$h}zwz-k;{SNy{Q~A+ANVpO1xDI&-AIv=CMYRV(l0A{zc2+OB^V(kAt-62Bm^aml%$}fladsabW&1+k{~4|D2e+@'
    '>eCqiA|t@A64;f*t`gXl#I6$9mBg+R*j2=?64+J5t`gW)#I6$9Rm83m*j2=?64+J5t`gW)#I6$9Rm83#u&an&Lts}CyN1B7B6bad'
    'T}A8~0=ovWYY6Nb#I7N*YY@AJz^*~;8Unipv1<tI8pN(4uxk*zhQO{t?3x0*2C-`j>>9+bDX?o0yQaXdLF}3WyC$(~3hbK1t|_o<'
    '61%3ru1V~g0=p)$YYOa|#I7l@YZAMrz^+N`S^~Q!v1<wJn#8Upuxk>#mcXt_>{<f57O`sy>{`UGC9rD|yOzMNMeJGvyB4u)3G7<L'
    't|hQ*5xbVau0`zH0=pKmYYXgJ#I7x{YZ1G)z^+B?+5)>av1<$L+QhCcuxk^$w!p4U?AijmHnD39?ApYxEwF17ySBit4eY8zVAlqA'
    ')giEJ1H0-F*tLOObqMU*z^*#3v+Jo}pD|K`5mFL@l155GP|`@rMRuM08uz!#&j&7VmA^M~FN8lBDZvOS>6ew<-zxuLq^=tQB_${+'
    'Qc{AF`x|HP9|%E7BPAgyX{00tC7qO{prn(M5|jifDM3lxS8{pd>^Z+aW8|um*1++tU3YwI;P}?AJH9n=d~4So-x@f+_3Msr4IJOP'
    ';`k&bO;A##qzOujlw6*)S^$MxT&Hjg8jK+f#)L_$<@HIcCB3c`UKh=btrR2R6Jcg-G&43~W^6PwHeqILG&43~W^6PwHe+Ti0^|??'
    'Sp>)-0I~>>LjdG}ulhEwf7Q1EU-fNV|Eh0;067Fe4r133*foe<Ltxh+b`61DgV;3$b`4_J6xcP0T~lD!Aa+fGU4z&)1$GT$*A&<_'
    'iCt4**CcjLfnAf>H3fD}V%HSdHHlqQVAmveO@UpLLip*)f<K&A-v032>#x54T|58vk8i$v{qmQ0-@pFhmzVEe|8)N8zyGbos?S@A'
    ')vWq#iB+F}zh*pMRL?%vCnZ+@cp%nC5B#%^>$ezLU?k(ZkxoidP|`_B2}**Ll%OQ;E4d}GuN%oABZ877B~4ILq@)Q-ij?%rN}l-j'
    '86!(DLP|nV(nv`NN*XCiK}jbiDJbcrqy!~FN=i@y<75cqWY9Pn!Z;Z;PKGc}291*;jFUm*WC-JA&^Q^wI2klfhA>VBjgujalR<O*'
    '6z2FrbNm$M_(6006z2FrSeXJ=CShd?Seb;CDPUz1R;GZJNm!WzRwiL(3Rszhm3bX2Suo;rTp#gC6i$J{Nfb_j!bucPfx=0EOaYKd'
    'fJ_09Pha^nMg%2AN-k44iNYyRI43!N367uS_$4@glH=EP#}|$G5=MN{h%aHp7kzh?@ZD8}l@hR0gq0GoQiPS7N^}(|xr~(}tkfN>'
    '+`3}r`A@&w6)WH8jq}C%XXm5Obj7;mTEA}lqX)js$Oa=luNpb|M@sq@jF6HLlr&Njf|5o`Qc%)KNeW6jDJelokdhLV#C;|E(-{9E'
    'Bbj7GP?Dsi2}+8TG(ky`lI!f+#I7x{YZJS+z^+Z~+5)>av1<$L+QhCcuxk^$w!p4U?AijmHn6K70=qV_>t;kylBA>wN{W;;K}nI4'
    '%j|mU*Jq4u!3Ze{K}jPeAt-62Bn2g%l%$}fladmY1Su&&39zeOXV+7|K4U~slBA>wN{W<RX4iT3`~7ipyX8Kw8@WGDZbmLENnqFQ'
    '>i4{E<UYG@Mg%2|l%$}flalK`DH!p$Yuod>k^3Y5W<*euq@)Q-ij*`#Ns*GvUn@`j`izk+7$GGgC~2f51SO4>q@bjek`$D5Qc{AF'
    'ASERz0d`G+U4z&)1$GT$*A&<_h+Xru?E3Z1EU5o*3H^tQ`VW`Tf4Hdsa0&f~i~0|j(0{n7|8NQYhl~0Tm(YK>sQ+*Y{fCSC50}t?'
    'xTxi62`x{HTAr5B^0cVsX-(yIbyAXo5*Um^Q_7;IlqEE!ENV(wOX$Z!)lY{|{d7?E(;-wp9aQ~v2-QyqRX-g<_0ut_AIlW_u}td6'
    'vV?vtllrl&>y?|wq^?*?=!!L|E7lUaVl8s~5*)wC@mp~GBFAsR@rxY41;=lya6W|BwW-4S5MI}&3g<(4T}4W+S2!P3gL()xs0Y>0'
    'A42{7LDAKoMAskw`ftDc%U5sT{d}>!wb4QNCLDxsbP&D?2jLqXgm1z@_(lien{W`mlTW(flTJ#yqNJ0O>y<)51tTF8j0E{41fK+r'
    'lMu#9P|Z^cHBUh`Pbt(q1=T#IQ1cX2*PKFKb5LPb3WZfk<0OS~k~B_I7$=XO43C}+kDd&Vo(zwk43C}+kDd&Vo(zwk43C}+kDd&V'
    'o(zwk43C}+kDd&Vo(zwk43C}+|7&_O-0niWuVf2KHq4B-``H8|jf`BLv~Jgh-ET5*2rVoQYGH8*Ei4Y|F42VU5{=p=wCf|jXbZ4}'
    'Ex@8Jz!J6qt3wE%4_psn+6gLQC#WcmUa1%Xp9pF6N7I@|)0#)qnn%-`N7I@|)0#)qnn%-`N7I@|)0#)qnn%-`N7I@|)0#)qnn%-`'
    'N7I`Bk2kG(>er`6uCuEVyPCkRM(k<=yBe{p3G8abt|qXn5xbheu0~jC0#+Jfr3+YTgq1E}r4d%TfR#pA=>k?7VWkUL>4cRoV5JjQ'
    'x`35VSm^>*I$@;?Sm}h7E?}k8Cs7wZi8`^X3+(E|t`OMOiCrPEs}s9IU{@!0g}|;(><WQhK_MJM2nU652q7F4!YPDsPza|G!a*UN'
    'LI?+ia0($D6v8QlaMJXf!t_h6D7(;#vQsO{F0`WT)QYkTttdOSqU=H|%BOyP`alUvf|Qh?1lUyqyOP*d0=tsfRRX(`*i{0%lGs%O'
    'yNcLV0=tUXRRX(;*i{0%ir7^GyNcLV0=tUXRRX(;*i{0%s3C9{8UlA}2;7B+z@7TScA+n9r@pXV=nLDaFKie3!glHl+l9Wco%+Id'
    'p)YKwzOY^B3)`tLY@bR9Cn;%yk|HHdP*SAi`t&<#`klh`J8Alz!t^_dT~lD!Bz8@KU6a@~1$NQt!7iL0>~wmt3#SJ=ogVDM>A}a9'
    '5|1k-9#={{u9SFODe?d2l@duC%qeUzCv7mNu)&<P!JNVdbJ7NL3LDHx8_X$eFehy=r?A1Cw85Oh26NH|a|#>GNgK>5Y%nKnFsHD='
    'oV3B5!Ul8F26GA<%t;%}DQqw&Z7`>>!JM?goWcfk(gt%18_Y=?%qeUzKlSS~Mz&ytl!Tz9k&+OUG*Xg+l1@reP|`_B2}**Ll%NFI'
    'C7disT9~G=Fil#RCM-;=?K<un`Ow6;UbL8P#xq7f^aegYdf<zUw1bS4V1$&Uprn(M6qIyQQi75oB_$|{`%3cG#s0F9JjjTkBuPmV'
    'loTmxf|4R7{j!oLetpJB2}Vds2ud0$2|-CCB`GNBq$C9;os^WIBuGgKN`PG@uq%mOC9o@rT_vz9iCrbID~Vktu&an&C9tcAT_v!q'
    'h+QSHtB74Cu&an&C9tcAT_v!qh+QSHtB74gU{?{lhQO{Ob`61DMeG^^yNcK~1a=K#*AUn>h+RWq*C2Kcfn9^xH3W7IV%HGZHHck9'
    'VAmja4S`*Q*fj-q4Pw_6*foe<Q()I1c1?j@gV;3%c1>c}6xcP1T~lD!Bz8@KU6a@~1$Iqh*A&<_iCt4**CcjLfnAf>wFGueV%HMb'
    'HHlqIVAmveErDH=*tG<9En?Ra*tLjVOJLU`b}fNji`caUb}eGp64<qfT}xosB6cl-U5nVY1$HfB*B02dDTKEW!ka>P3n9EIgtrjF'
    'n?iUCA-pMsw-CY`LO2c~gg1n697+f$DQSX|A|*{wQl#Ya^!wDW&lsueMuLo#VdVbnYS%;fGy);qu7~jXGa4zm49N3mbW)Onl1@rW'
    'P!go11SJq%U5Ktm(bbjcN>b7UB}Gb_prlC2bwGBCt}aAZr|1eHx;jNy2+`Fkx<ZJqPSF)YbajfZ5TYw+`VC?FrL#mKoFxi6OBBLc'
    'qM)-xA)F-&I!hG7S)!n`L?N6d3OY*^!darAvqT}BB?>x-5yC-?pv}w>HZy}ZGeg+S4BE^LVKXymGc$zE%%IK85H>S|HZw!m%naJh'
    '3}G`fXeTIyouHtdpb&O~f_8#J*a-^C)rXL)56abtkgE^M)rXL)56abtkgE^M)rXL)56abtkgE^M)rXL)56abtkgE^M)rXL)56abt'
    'kgE^M)rXL)56abtkgI>{*QZ9BprlAi6O<Gwxz4Ul?AijmHnD39?4nJY5H@LoHfciGqzT%j31O2aXp<&{O`4!hnh-W=f;MSV*rW;C'
    'q)B0uCTNo;g-x2EO_~%oX@WLsQrM(P+N4Qglji&x;FEUUCrP>b6ms=Rx%w1x^+~z<6ms=Rx%%viPl}W@K}nI4>*J);IO)PTp$&l)'
    'HUyG31X9=#NGi!qp(Hn{BsYbU+@zA+6iRZF(&(9rPXH@I8a*kEo<bTuDUF^&8a*kEo<bTuDUF^&8a*kEo<bTusU$arlH8;!z!a(g'
    'ld1qyr~*u?0?blgS0g1UDCwjm1tp!7Tt<D-XKD$bsi%?q?5cL1U3CDvs$FMSH5iQ5t`Ej)Fc_;{AB<1^`iv1lNh2jGDCwjm1tp!7'
    'Tpx^`e9{G<bn;0TeA3A$UGPaKpLE41K}t$ca{pVng!M_$fu#}-EEOGCD&fFV(SfBB4lETNSSsPbQqh5>5)LdCwbm-3wN_DUtrA*m'
    '6}8qXp|w`gI=()?j&E(AJ=G$P&-GMmzi;#Jz4DG1Gta-VJ~Hx;2i`AW{`G+`Gg4rr9oLN%DQSX|A|?H@lJ^T!Fj9gMQWAoaMoL0Z'
    '(nv`PN;)Y?K}jbiB`66}Qi77WucSVW@h>t0>?(m>N$e_tT}kXJfn7=LDuG=^>?(m>MeHhpT}A9Ffn7!HDuG=^>?(m>MeHhpT}A9F'
    'fn7!H8UnkD*fj)p6|rjw>?&f{5ZG13t|72%5W9xJu0iY?0=ovWYY6Nb#I7N*YY@AJz^*~;8Unipv1<tI8pN(Cuxk*zrogU2?3x0*'
    '2C-`j>>9+bDX?o2yQaXdN$i>eyC$(~3hbK1t|_o<61%3ru1V~g0=p)$YYOa|#I7Z<YZAMbz^+N`S^~Q!v1<wJn#8Upuxk;!mcXt>'
    '>{<f57O`sy>{`UGC9rD|yOzMNMeJGvyB4u)3G7<Lt}U=@5xcg)u0`zH0=pKmYYXgJ#I7x{YZJS+z^+Z~+5)>av1<$L+QhCcuxk^$'
    'w!p4U?AijmHnD39?ApMtIs|rYU{@UiyEd?^4uM@8*j0zXt_|#}<2t*Z`t=zjB^V(kAt-62Bm^amlw4+4QC_%&yl_!oxP-iLQC_%&'
    'yl_!oxP-iLQC_%&yl_!oxP-iLQC_%&yl_!oxP-iLQ7Wp0R8&zas)STjQ7WpY^13=HNkIt=Mj=^Hlq@JASx}TLsLRQMa|`?asi^aT'
    '%cr8=8@Wf}4@OEbLQ48&CHGH7eK1njjewF8loTl`LCO6y67L@fK}jPeAt-62Bn2g%l%$}fladmY1Su&&N!(X*`HaMKetpKsWhH}r'
    'G6bIt^2rc<GHEbQVK7b_j8hnllLq4y2IC~hPr>n%96uGu!(d#(V4O4<moOM7eG)C<lW5W>(Got17CC+ij$h>XEjWIW<G0}WMULNs'
    '<2Ssnb_lO)!|Q5?@VYj<u68J|t4PV^ancT;a67J3xE(MU+i`s`wu7RpU5~ETU}kLX`pnoG`J@RxY2=e8_@t3fy5bWU@%{RUf9lt#'
    'Mw+0cNJ$fv6e+p9#BKqfsV%O5rnaEL7{Xvo_^NMt{j0tuy{;5q7p>!4DMr92!aBauI=%_(_(tpaCamKdt>c@pj&HP%Z^Amh6#;Sx'
    'fGh&!5CB;O$RPl7Kn}8v>p93aAP3pT^&Dgy1jr!(auB<Qz^*~;8Unj0-_?YCSEGDa6Y^bc5WA+pu0iaY0=ovWYYOZd#I7l@YY@Ap'
    'z^+N`ngY8fv1<zKqLghDQnrm!woORcHcHtxA!XYrW!r?5ZJQLrPj|8Za9Vl$!*{R0`ucbM{MSFe`R?`0U*3KH`iEa$zI*-C`KSN>'
    'x85Ip-g<w`s?YZR==1NlRK<(x*~j{%_s8EJI2SnUqX+)kVC1(LSzx5cRU>E97gExfV1$%}prnzKxU3{eNeW7ml%%4hk&+aYbW)On'
    'l1@tMvXY{4Qo=YXimno(t0=nadUV~Iyx&!F>w6y8jdU_nf{_R^BD}7+ucUn%eqUrHgNz7Dl9V(-Ns*DuGh@>9o5J*)LD3aLbOl9M'
    '3ego5T`5FYP;{jbT|tgd!SM<6N#^J0lUs|rJpbu;Th!(4ZoAqq&ObXJeWpcSzQ30)KYHK`jGTWh$;cLrkdhFTG*S|Rl155WP|`_B'
    '3Q9UDDM3k)k`k1}eI@JD82=(8nPfyzlBA>wN{W;;K}nI4>+D*@t|hQ*5xbVau0`xx0=pJDehZFY<oGQ(ev{+3;P_3B--6>eIerU{'
    '-!y6M>yy@PId@mdZEdjyBO8pApd?612}<ryTDRq#pyd9fbu%I;Nm9}TB}Ga;W!E46`ftDc%U5sT{d_H3+#95I&Ic~1b$Wxe&dtaY'
    'jF6HPlyp*ZIceBC`J@Xz>7=A9N;)aIoCWK&P0)pHf==56UDzh*v`x^3ZGui2mo8*nI%QnCka6jhap^+F<up>HqzOujlw3~hbV}>='
    '6vhc{LUmyis?#P^7dD|fZ9;Wn6ROiDR2Md(I&DIAVH4`9U!O4|C;@hrz^){AmB6kfc9p=cBzBd+t|WGqz%JV2?7|jjr!CGdY;ktl'
    ';_Si}=TpBveINxTos^`Yq?3{olmsa$K?#hLA&iqj<75cqWY9Pn!Z;Z;PKGc}291*;jFUm*WC-JA&^Q^wI2klfhA>VB&GA#1;|I<0'
    'Q<&oi&GA#1;|F173Rszhl__9l5>}>wl}T8c0#+trWeQlCgq0~^WfE4VaDMG^V)b!i^>JeLabopxV)b!i^>JeLabopxV)b!i^?%)o'
    ')n9F-$nisPJS7WKNERd|3sOiHBqa+{NERfm<5O72C#~aCSjRto<<A%qloTnsJ{T!kkV3K`Cpmrzj-TZCB{+VP<JWb^SBEdi^MT7>'
    'kVOmA5*DUKYsN~&$o;SS64s1G6fS|nMHDW9!bK_D5>mEBs4oHaMW`<U^+l*J0rf?wF9G#MsIPgwQmE09{w5shZ*-);2}k-H9qDhv'
    'k^V+U`kQd1ztNHYCLHO1>er_aq@bjek`$D5Qc{AFASERzxqq^|?dvDIpZfI~BZ877B~4ILq@)Q-ij-Vu7geP+p(>?ORZ0`8QW{mI'
    'G@&Y`QB_J4s#2c%^%*0Al1@rWP!go11SR*`)enJP8`yO-A}C2x(gY<%N}8agNXcb(J@xA|Mz&ytl!Tz9k&+OUG*Xg+l1@reP|`_B'
    '2}**Ll%NFIB~;XPs;KKiMO~+gx-L}Ib*iZA?K-=j`t_-i>+EX8t|qXn5xbheu14%?0=pWqs|oCC#I7c=s}WY3fR#pA=>k?7VWkUL'
    'X@r$7V5JdOx`35NSm^>*I$@;?Sm}h7E?}h-R=R+dPFU#zRytv&3s~v&Nz{c;qE77U0=qh~D+G3RVpj<4>cp-P*wu+$A+W0xyFy@B'
    'PzZ+*!a*S%LI?+ia0($D6v8Qla8L-R5W+zroI(f(g>VWXoHYHWF#RS?zbQ<=Nz-o%({IxBo5J*)H2tP9{U)(11$HH|D+P8Xv8x1j'
    'C9$gnb|tZ^1a>8{s|0o>v8x1j6|t)Xb``O!1a=j%s|0oxv8x1j6|t)Xb``O!1a=j%s|0oxv1<tIDq`0V*fl7GhY-SpLU;%vJSc>R'
    '5W<5(cnBdpD1?U)!h=G13L!iwgr^e1NlKcaq)15<loTnsKK)Wp>n`-P?$pz|3q7qn^|bCnPwP%St-H|Ex>HZ<F7&kSbb7E0rw2Qo'
    '9_+&D!A_?KyU`Fhs4r{?&CG+EnTOEKoYwIntmA{$@gc0^gVymOtmA{$@gc0^gT5d`_<{`jf(+pcGK$)=51}o4QFrbkbmuPW&OL<g'
    '+=IGv51~8vpb<ZW5l>5p5S9)>ONS7a4na$Y5S9)>ONS7a4na$Y5S9)>ONS7a4na$Y5S9)>ONS7a4na$Y5S9*)>u?^|;XJOxd0dC{'
    'xDMxW9nRxAoX2%IkLz$A*Wo;_!};~s;e3%1QgWGHNr(4SIJ}>9ct3^1`$>oQQ#ibzba+37!~02x_ft5$|J1KfA1FadkdhLV0K0?='
    'fust76e<LgDg;uf5J;*JNTEU?sX`!y3W1~wffOnPk}3pJs1Qi15J;gyAgMwig$jYB3V{?V1d=KQQm7C}st`z_LLjL^AcYEnqzZu)'
    'Dg=@$1X8FFNU9J>p+X?3LLh|-fv0|b#>f_okdhFTG*S|Rl155WP|`_B3Q9UDDM3k)k`j~vyM)q^q>}|HoGeH>S&(qDpxUnEu8|M-'
    'GuDe1v(0$M$cJkWkB=VsA|ve}BPAFiB`GNBq$C9;os^WIBuGgKO5(nfb07&y@*pFEk|ZTfP*S9%2}+8T^vg<~`1KhhB^V(kAt-62'
    'Bm^aml%$}fladsabW&1+k{~4|C;@hrz^){AmB6kfc9p=cBzBd+t|WGqz^)>8mB6kdc9p=cB6gL)t|E4oz^)>8mB6kdc9p=cB6gL)'
    't|E2~fn7!H8UnkD*fj)p6|rjw>?&f{5ZE<{T|;2kAa)IbU4z&)1a=K#*AUn>h+RWq*C2Kcfn9^xH3W7IV%HSdHHckPVAmjaO@Up5'
    '*fj-q4Pw_6*foh=Q()I5c1?j@lh`!{c1>c}6xcP1T~lD!Bz8@KU6a@~1$Iqh*Amz@iCs%z*Cci=fnAf>wFGueV%HMbwTN9yVAmpc'
    'ErDH&*tG<9En?Ra*tLjVOJLU`b}fNji`caUb}eGp7TC3jU0Yz+rV!pj2yY7EErjr<5Z*!vZwlcpgz%;i-a-g(2;n${5Z(~NHzR_Q'
    'BqdEyQlz8_N{W<Ro_?SD^%)~|-AIs;GK}1RUF~`ZpGF{r+w~AWe?}uEmjQYHj7~~YP|`_B2}**Ll%NEns|(T9D7v~5T}eusprlAi'
    '6O<Gwxemxq(ba|M>J(ieL|3Qi3L&~WMOO&X)hW6{h^|i26+(0cO}`;bzjT%;gtJ6JXNf{MOB8gLD1@^_L1&3VI7<|CmMDa?L_ueX'
    'LO4qlbe1TDvqV7$F+w<q5ww{Z!e(aBW@ZSRnL(SGA#7#_ZDxkBnHjX18Ny~}&}L=`o0&nInIUXu2JHleuoD!t6BNQuP|!|L2s=S_'
    '09LA9$I5BsK31w-$4VW5m1@_qQVpW3+V$wF2GLdRdUQSY>oZ0KC5@D%prn(M6qIyQa(NwJoqW;-pLFs`7ktvmCtdJKC!ci1CqYU|'
    'P;!5LQo`!DX!Tpd>bGe1Tf*wMX!Tpd>bGe1Tf*wMX!Tpd>bIy)tAskOqB^Y->a>dLv`VPcDyq|}&(~?SHqQdRh~skwdhPdZ-s<n_'
    '{Q=mS=igW#8TrQp@7EUp`oNbNDKOHG>qd%{G(ky`l73mq`!ypNDZvOS2|-CCB_Swjq$DpZne@7*@VX|wt|`2(Nv~@vud9=i6qLYV'
    'T*6>nG#HmK7#Bs?x*A<)t<HVNzkeVFBljHt!AJ>4NJ$Awf|Qh?B<?GD5`JG~Bo8toC`nS%1SLgE+GQn+9KQs|FLL}A9KXo%TX6g$'
    '$8W*$8(vpCgx9s<b+toyT^n9kJCxT|q~!8AX$Mfa9oH%R{-n=cBXxZ+zTXVGYh+xHu2vLX<9c+pqUajeqpKA~*ASxX{z~h;k$zdp'
    '{gu`SBPAFiB_Swjq$C6-jg+LIq?3{qlyp*3f|4L5B`5)QO@Up5*fj-q4Pw_6*foe<Q()I1c1?j@lh`!{c1>c}6xcP1T~lBet$v%Z'
    '`farOZNloe(dxGetKT*$grDYC{%~4(`@?syzxw)jasKNc-+cG_<uC8PfBnNRFW<fX>HO1w|62yU&szq3R(&=D-rw8Eb7`?&RL?%v'
    'CmHa+J#a29;-d%tnP&ghM$R=5841BiBO`g$$a(Uh-!*bOnjjeY<ZV1-WL-Cs_l>kqo%RJr&YwX>x?yC2k;~C_JKjJ_;&V!V^)Vo-'
    '+zg>|GpKSigv!mJ%FPfeH-joSL#W&gs@x2rax<uMGla^`pvuh<DmQ~FH$$l0eCpSyMw+0cNJ$fv6e+oWKz7qP$t|3d+{CUeu!~A|'
    'Lnzq|D%lO8WH+c}H-wVippxAXN_K-vc0(xH4Jz49p=39xWH*J9-Jp`)6iRl3N_JBy*$pb$O`&8rsbn{WlHH_}(J7paPO98Yp>i{+'
    'ax;a>&7{iB6e>59DmSw$J}FYt1SLgEu8)&W<D?7Ygbv82a6mTcfNTl}WRnj5rO==~sX==R4ce0$w5QOZJ*lWNQ}GF4MJTFFDymGO'
    's4}UjGKHebq@v0ciYk+eDpM$`Oe(5Op{O#cL3;`f+LL-hr_d8RsV8&_J)x6&LT4$jtC5lvlyp*(f|5>3K8N~WeT4A?AnzPu{4kKN'
    'os;|k$oobvA0+<(NH8KODN=Iz?CJ+Vf{_x8kdhFTG*S|Rl155WP|`_B3Q9UDDM3k)k`k1}eI<4Ec*Apkea48OBuPmVloTmxf|4R7'
    '*V$FXt`gW)#I6$9Rm83m*j2=?64+J5t`gW)#I6$9Rm83m*j2=?A+U=I_);j~ODf<?p@1)`fG>ptzN7-a6bkr~3iwhe;7cmtOQC=-'
    'semtq0=}dIz7z`hauB<Qz^*~;8Unipv1<tI8pN(Cuxk*zrogU2?3x0*2C-`j>>9+bDX?o2yQaV{s-{h$nl`DLHic^1q-xp}s%ev|'
    'X;Y}C%}MN<0=p)$YYOa|#I7Z<YZAMbz^+N`S^~Q!v1<wJqFU_~YPFMUwNt3oPO8<;C9rD|yOzMNMeJGvyB4u)3G7<Lt|hQ*5xbVa'
    'u0`zH0=pKmYYXgJ#I7x{YZ1G)z^+B?+5)>av1<$L+QhCcuxk^$w!p4U?AijmHnD39?ApYxEwF17ySBit4eY8zVArOGRfoW?O%1CK'
    'fnA##RviMnHucCkgdRBu^~gDd9ytf~$T@@#Ob2yfI)n~P2X$aNgho6;Z8=itGZEB}C53)00i}pXUazJ-5~^vByk1RvBq=FDNs*Eg'
    'loZqjA0;%bDyR!S>Uv%9QPdTyguz(U6|01<SVdj2O6ZD3{a6m6AIm}gSPr2d%R&8E4xt~*L476;q0huYeI^d0&qSlr_9m3JH!5v!'
    'LTP)Wwj51)UC@@J3vD?XwdLqSTaHF;Il9o6qfuLqF0|!n)YG5~Z8?JaOoY&9B4{v%Fc_2Ca-`6fBk6TLJ#+Vmzy90r{_@q^cRye2'
    'F43tpzYC@Lol5ijoznc@VuX~Wprn(MysV^?PrBffPD;9>q?3}%%{Y2M3ybrC%PlN!Miv+eK}jPeAt;IaO7he9`9((V_b#~^5tJk;'
    'X@Zg>B~4ILq@-U~k~B_I7$-^NB!zL3G)__&CrR^33iC-4g;St#5`|Nsa6(I@n-M_?u&V@iC9$gnb|tZ^1a>8{s|0o>v8x1j6|t)X'
    'b``O!1a=j%s|0oxv8x1j6|t)Xb``O!1a=j%s|0oxjgujalR@KT2;*eXI2pn?88l9YFir-IlOc?gLE~fy<7Ch{8NxUjG){&vP6o~K'
    'Q<&oi&GA#1;|I<0Q<&oiVPy(fnS_-oU}X|krht`6SeXJ=CShd?Seb;CDPUz1R_1lAM2FAR^LFyfpQ$$^3yfSICqb#G5K>V=si+WA'
    'Q9-Gw5K>V=$$}7)1wqMzNJU8_B`GNBq$C9;os?W3@wA)^VL2DHoC{$&7qpxUVL5l}1$e)4_HjMBk{V~H&^SA(adrxgvu{T3D{0ri'
    'Ad|Xtr_i1IW`vY9K}nI3%QIur^qa!;n?cbPLUhsoNecTXN&6=$?4Km<pQNyVlC-sv!q!I8)<z0j8(HM|Avm6r1t}y8l9B}}Bny&~'
    '1t}y8lGgDltmBi`@hPn1pT6>Ej0j4Klw2Q-lq^UgS&)+)zXZoma{Lk;Kgsb+aQx#Ena3qEk4t3!=UyW7tBp{GyMzpP(ac!F%vdxt'
    'mM}9G&5R|?j72kJ2{U8S%vg73##^J)Jpbu;8=dC+OUv@b`Df>&&onx{C91w|`=bZG%*X~KJ+B%$r*Klzw_t>ngrKC6k`R<MQj&s_'
    'PD)Zx(n(1PN`jP>pd{`q*`LPv7a7SUBZ877B~4ILq@)Q-ij-Vu*CuvtfnA%}wFP!<V%HYfwTWF@VAm#gZGl~z*tG?AZDQ9J*tLOO'
    '{Ser-fn7Hvf|4X9O;A##qzOujlw4-lQ@=iAWD7<}NeD_BDG5PIBPA&)>7*nDC7qO%pd?612}*!n?K-=j`t=zjf|4X9O;A##qzOuj'
    'lw4<5BX%``U5(h)1a>uIR}<LPh+R!!S0i>cfnANT(gdtD!b%sg(g-VEz)B;mbO9@ku+jyrG{Q<3u+j-DUBF5wtaJe@ov_jctaQRk'
    '7qHR^D_y`!r%$3Td=hnHR~OjTiCrPEs}s9IU{@!0g}|;(><WQho!AuuyMjVEgb)r2;SfSND1=i8;h+#sA%ufMIE4@n3gHw&I4Fcu'
    '2;rpZH-+gpY5GlJ`c0aCQ<#2}rr#8%-=yg`h3PkmT`90DiCrnMD~Vktuq%mOC9o@rT_vz9iCrbID~Vktu&an&C9tcAT_v!qh+QSH'
    'tB74Cu&an&C9tcAT_v!qh+QSHtB74gU{?{lhQO{tAv}Z-9u&eu2;o5?JcJM)6v9IY;Xxrhgb*GS!cz$0K_NVq5KdCk1SLgEnxLde'
    '$@S@X()2rp>37ofJB8_Y61%3ru1V~g0=p)$YYOb5(}P_&J=p2=U>8mgb~-)Sh0}u(;P{B^j*kGxM_hM&&^kVZb$rk|K7@6A&^kVZ'
    'b$rk|K7@6A&=+I~UywmxkRg0QM#1zO<NEX)1=DYg>(g%(OusR%Prpz7`izk+7$GGgC~2f51SO4>q@bjek`$D5Qc{AFASERz0d@&X'
    'hoGfH2up{cr9%iyhoGfH2up{cr9%iyhoGfH2up{cr9%iyhoGfH2up{cr9%iyhoGfH2up{jetr5t2}**Ll%NFIwFGueV%HMbHHlqI'
    'U>DWSg-|;eR67?!?Of36H-y!1(CRmY)o;-1H;mOU6&{69cobB46hh%qP~lMsg-1b!M<Em*1r;8JP<Rwncoa~0RBhLB*T{#<BkDzq'
    '*=9Ur<ini|$43u*k&$+gkrIrMk`$D5Qj&s_PD)Bp5~QRAC2?QL`)eI98_9!=2uhNaG(ky`k|roAQqnIgdE(b+jFe!6l!Tz9k&+OU'
    'G*Xg+l1@reP|`_B2}**Ll%NFIRRX(`*i{0%lGs%OyOP*d0=tsfRRX(;*i{0%ir7^GyNcLV0=tUXRRX(;*j3lr6%CX`yRIY}D2b*h'
    '>7=9tB|%C`P;#Hb(XUhZsb8NlA}C2x(gY<%N}8agNXcaipFbl=$@S>!6kQ=iSEuL-A-Xz6R|wJ7DY`<4u1?VvLUaYp@e$YO_%o>n'
    'KG}j#7WrfgK3U|GE%;=UPqyHb4c}e2%$=a5k&+OUG*Xg+l1@reP|`_B2}**Ll%VAPd~!ClKLjQB=aZWeK}nL5CMYRV(gY<%N-neO'
    'sb8NlQrC@uPug{#Bz@JV@KvAmRiDCFebQHb3Sad}U-j7)pA;!+f|4R7*T+exanglxLQ97fmJUfvhZL3$N!v{+Y&RusH>I%Ml(gNH'
    '!gf>Ac2lO}6TphF-ITQ5l)`pX(sok{+f7N^O(|?QC2cpQu-%lj-IT(1Q_^-*3foOd8SWG^+({Yk6f)dN8SX6Qbv06wf|5>3Qc%)K'
    '$z{|ReWsT1nR>f`;vgd>7$GH>*>$@-f|Ohyj5i~sBm^aml!Tz9k&+aYbW)Onl1@sl55`VD>4Hx>`J@Xz>Ex3x_@t9hy5f@{B_$}i'
    '|1Dg?`lM)UqlB%EqOFY*wl<2kHcHsqDB9X6VQZskYomm%jiO_tB^(<qIyPFuvC*Pqqa_?0En3Id=hyMAomco|#PPZF!tM8M-i`x*'
    'SeVW{|Hj8x`26F6_Y0VRec;QC6c}m8bt6SenxLdeNx!V*{lXNClwgFEgrKC6k`R<MQj&s_PD)Zx(n(1PN`jP>pd{`qsZV44i;Muf'
    'N?=zKyGmeJ61z%ZR}#BQU{?{lN?=zJyGmeJ5xYuYR}s5PU{?{lN?=zJyGmeJ5xYuYR}s60z^)>84S`)n>>2{Qir6&-b``N}2<#ff'
    't|72%5W9xJu0iY?0=ovWYY6Nb#I7N*YY@AJz^*~;8Unipv1<zK8pN(Cuxk*zrogU2?3x0*2C-`j?3%=`DX?o2yQaXdN$i>eyC$(~'
    '3hbK1t|_o<61%3ru1V~g0=p)$YYFU{#I7Z<YZAMbz^+N`S^~Q!v1<wJTEwm;uxk;!mcXt>>{<f57O`sy>{`UGC9rD|yOzMNMeJGv'
    'yB4u)3+!6Nt}U=@5xcg)u0`zH0=pKmYYXh!#I7x{YZJS+z^+Z~+5)>av1<$L+QhCcuxk^$w!p4U?AijmHn6J>fn6KeRfoW?4eY8z'
    'VAlqA)giEJ1H0<D&aS6^ea1)$Mo38rN*XB%K}jPem)TX67cL<$T$C3sAun8%7cL<$T$C3sAun8%7cL<$T$C3sAun8%7cL<$T$C3s'
    'Aun8%iYg%$Rg{V<Ar)1WimIu+u1-o)Py&NdNEQ?&3ra{96eSDla<bsu!oGhh>UPcARSLg1a*x6vjFe!6l=RC=?w^YKV5F`a0VO3U'
    'DN<5`lKW>Q-ain6l155GP|`?A3Q9UDNkK^`B_${cQc{AFxUb~$8Hwlo`izmwN(T942tFC)lOgzI(qNpzV4O4<r!W{N4aO-9#z~H!'
    'g5xJSekzWK!MKFMIB760VK7emBwE5J(WFnJC43Ssa{Lk;zsT`haQq_2Z^7}49KQv}Z+Knp5MI}Y*VPW;b!~WE?NDA<k&?^fq#Z!v'
    'c3h`$J76%j<N9E12Sr!A9$l@$%-GuXnXxtUNfUh1$R|zkNh6<h#V0W0`}Gn3)UQvCG(ky`k|roAQgV5T-2y&STU`H4Z9#)Egu$5b'
    'Rp0XZSA9!*T~AA>{%~4(`@?syzxw)jdH(Ak-+cG_<uC8PfBnNRFW<fX>HO1w|69FdpSOC)S@qd^$3FjF5WZei&py^C^^X5|AmgJ4'
    '{#jJ{tBssBsP~PW<)N32BpFG?NDDHuK4aw9UMF*QkGp@x&)ER%@)bYl&$xfZ&&|jdjF6HLlr&Njf|5o`Qc%)KNeW6jDJelokdhLV'
    '+`pIZ*27Lva{pesn-M`tl9DDUDN@n|B}GcEv#S%ky1=eZ?CJu$I<cz@?CQj>F0iW;ySl)xPVDLeyE?I}3+(E|t`OMOiCrPEs}s9I'
    'U{@!0g}|;(><WQhLF@{FT|w*$fn7oD3V~fg><WQhLF@{FT|w*$fn7oD3V~fg>`H-MNg<p<2q%Sb-VfnlbtBTC3#5i{fz+T2q=s;T'
    ')SwHbhH!z@pbMmiaQWS!%kPG8`Q4z)?}l*s-Jr|wrf~V)pv&*3aQWS!%kQRe`Q4z)@1}71-Jr|wrf~V)q|5K7aQWRNc1?j@lh`!{'
    'c1>c}6xcP1T~lD!Bz8@KU6a@~1$Iqh*Amz@iCs%z*Cci=fnAf>wFGueV%HMbwTN9yVAmpcErDH&*tG<9En?Ra*tLjVOJLU`b}fNj'
    'i`caUb}eGp7TC3jU0Yxm-GSDHJJ34af!2jP&^q0L)`dIJI^BWRg*(tX-GSDHJJ34affm9YXr1mri$e+FBqdEyQlz8_N{W<RzrT_0'
    'Knvjxw4h5^Lb!w_=)&|6E=-Ri#}DE1yCTOA!SRC}KLp29zAJ=$S5UqygnU;}zAJ=$S5UqygnU;}vLJ+HK~S<Fgk(WbvLJ+HLC|t8'
    'gyme&axR4BT+nhZgyme&axOl<ocs01<)|X92^DFLD$<%zk=CdptqB!rjVjWbP?6TCBCQD(X^kq<noyC}ijMjY;izxXQQsjP^({K;'
    'JA|XY1FC7;xL!@$22|6walM+h4La&OgrmNL*fj)p4Pw_2*hQ6_O{m;#RJqxN%FQ;2T~lD!Aa+fGU4z&)1$GT$*A&<_h+R`)*CcjL'
    'fnAf>H3fE2@o^K1j~f*qH=+2rQSosTijNx=A2*@+xJ?S-r~O#|@Yh>i$Dd#Q`18AO-~IX3_dmaUU)b@(fBt-{==jT5Z{PiVt;(Y}'
    'sEs%uxLg}?{*3#z5jP`CFhWXFP|`_BURKh{CtdJKCna4`(n(2OR&pL`yssoa{Vl%C$O0oFC~2f51SN4_Nq+i1zsN`i84;8uDQSX|'
    'A|*{wQlz9`R+2PMQWz&xYuANZyH2%sJ%w?SG@qm}pCnN@1qvroI0Xtn@#`~21SP<(64;f*t`gXl#I6$9mBg+R*p<Yt64+J5t`gW)'
    '#I6$9Rm83m*j2=?64+J5t`gW)#I6$9Rm83m*i|%6hA>VBjgujalR@KT2;*eXI2pn?88l9YFir-IlOc?gLE~fy<7Ch{8NxUjG{;Y2'
    'jvq9~PhpN9G{;Y2jvs`TDPUz1R;GZJNm!WzRwiL(3Rszhl__9l5>}>wl}T8c*Rc{EK2syEf2Q7yEHH9;oCKw!LP$jgrJ_PeMFpjz'
    'LP$jgB@04G76c^=A{8Z#l%$}fladsabW(DC#M5#vgyme&axR4BT+nhZgyr1X_UV3foex}&uJdPfGE#z(`(KbZBf{&t{{?w#J|!r*'
    '{{@-t`WNKQ2q|fTk|HCQXU3%IH-+gpgQ6>h=%W3T6!uS&_D@pSKS|m@Nn!saX=@{et&OCujTE*vvdHm6a6BaoQb-mgB@0qW79=GL'
    'Qb-mgt>aTz$0x1hQ&`78edW&>5tI}uxjq;vS&%}qASXF~367uS_$4@glH-@)_(e*#pk%`b_w8y+!AK(`*W1$4w{QvH!bRW0C437P'
    'eWsT1nOY4-{P{pPj4Uv6y^AlMR4d`6+M~1b|GLh~zuE|W5|!{tRP;$y!Y5JD{z(b@Cq>^~C46@keRq}c-BtA6Rl;{y(RWt~-(5xD'
    'U3KTX>(}<hyB%V_+ZXRV#C*Bk=FP|kBbVE4o<HLtC70W6-i(lv5R^1h5`vOON>WhLNl6MyIw>hZNsy8fl-zH(+4l8zn@|1vj1fUe'
    'l9DDUDN@n|B}GcEvuhK(w!p4U?AijmHnD39?ApYxEwF17ySBitP3+nNyEd_F3+&p!u6_vY+Q6=x5kX0kk|roAQqlw^MM^HS>#1L#'
    'F|q|Cq$C6-jg*9-q>+*olyp*(f|5>3N>CD{qy#0vu6CVWPyPCg5kX0kk|roAQqlw^MM|!-i>}7%!qr%vuEy%Z)mWXb#_Gb=Se>rM'
    '>cZ7novy~}!qr%vZmsIVtyP_Ft?I(9Rh@3F>cXv6oo=n_!mU-EZmsIVtyP_Ft?I(9Rh@3F>cXv6oo=n_!mU-EZmsIVtyP_Ft?I(9'
    'Rh@3F>cXv6oo=n_!mU-EZmsIVtyP_Ft?I(9Rh@3F>cXv6oo=n_!mU-EZmsIVtyP_Ft?I(9Rh=#`>cZtkoh~ox!sSJsE-&iB<wZRx'
    'gi{FNpb$<Wgo8pjg%A!3;S@qRY5GlJ`lVZ|x^Qb%r(3If3e#`W^qa!;n>78VF#RU6D+P8Xu`303C9$gnb|tZ^1a>8{s|0o>v8x1j'
    'C9$gnb``O!1a=j%s|0oxv8x1j6|t)Xb``O!1a=j%s|0oxv8x1j6|rjw>?&f{5ZE;+gohBqgF<)+Av`FAhY-SpLU;%vJSc>R5W<5('
    'cnTprD1@gH!bwV+prlAi6O<Gwxjy|)ntrD+{Z5*Gr!f6aV%HSdHHlqQVAmveO@Uo>daw(p2Roe}?8526PNxUEaC$K4vhENr>khiC'
    'JA})+gVymOtmA{$@gc0^gVymOtmA{$@gc0^gT5d`_<{`jf(+pcG76^O7}uxYD42d@T%UfUVET=5efoXs*Jq4u!3Ze{K}jPeAt-62'
    'Bn2g%l%$}fladmY1Su&&39w68Is`2pLRdNkEgeEwIs`2pLRdNkEgeEwIs`2pLRdNkEgeEwIs`2pLRdNkEgeEwIs`2pLRdOH_3P6I'
    'N>CD{qy#0vt|hQ*61$ebu1V}#0=uYoE`-{-pxU_*YUhGhzagxCgI2#GtbT)5zhSI?sqiR-!lR(VqYw&@f(nm9C_D-(JPM)kD5&r#'
    'gu<hs!lMuhkCG1Wr*L>b>F|CEhxd~X@27BhKk4v(3WxWT4)3RMct7dzehP>8lMe5vaCkrI@O}!1_md9qr*L>b>F|CEhxd~X@27Bh'
    'Kk4v(3WxWT4)3RMc>k$ipFU86k{~4|C;@f}6#_{W0x47oBvlBcP$7_1A&^3aKvIQ33KarL6#^+#2qaYqq);J{R3VT;g+Nk;KnfKC'
    'NfiPqR0t$h2&7OUkW?X%LWMw5g+K}w0!bADDO3m~RS2X|A&^udkV1t(QiVVY6#`HF`izk+7$GGgC~2f51SO4>q@bjek`$D5Qc{AF'
    'ASERz0d@(cAxS3-QaD+Vbh02nf3o0L-gNvSc<$WU{6XrqJDWckDKK*JZu1YpLrR*Uq)17>tR(N=ZT`VX2}Vds2ud0$2|-CCB`GNB'
    'q$C9;os^WIBuGgKO5(nf`ZUJB$Oy2j1a>8{s|0o>v8x1jC9$gnb``O!1a=j%s|0oxv8x1j6|t)Xb``O!1a=j%s|0oxv8x1j6|rjw'
    '>?&f{5ZG13t|73ih+RWqR}s60z^*~;8Unipv1<tI8pN(4uxk*zhQO{t>>2{Q2C-`h>>9+bA+T!@yQaXdLF}3Wy9Tjq3hWxht|_o<'
    '5WA+pu1V~g0=p)$YYOa|#I7l@YZAMrz^+N`ngY8fv1<zKn#8Utuxk>#mcXt_>{<f5Cb4S??3%=`C9rD}yOzMNMeJGvyB4u)3G7<L'
    't|hQ*5xbVau0`xx0=pKmYYFUH#I7Z<YZ1G)z^+B?+5)>4v1<$LTEwm`uxk;!w!p4U?AijmHnD39?ApYxEwF17ySBitP3+nNyEd_F'
    '3+&p&t}U=@1G~<P=?+23eRkc92uhNaG(ky`k|roAQgWGHPyPCgkrIrMk`R<MQWAoaMoKQTt0*sALSDEiFI+-ixF|1NLSDEiFI+-i'
    'xF|1NLSDEiFI+-ixF|1NLSDEiFI+-ixF{7>LMo~#6;(niswfpzQ+ZvTl%${p2BVNHC`uNTkSr)l7S!cr!MTNf|5VhiTjW&=zc+G^'
    '!XJ#3V1$(P%S!H_iuz!rt{VX*B`7IUQi78EXC&S~5Q36MN<vW5NJ$DxIw?s(Nhc*GC<#(hf|9te<nkGb=luGNk;_U3`D6$_8RU~8'
    '_+-*xoWfw7G#IBa7$*(JDGbI*j-P_#Cpms9j)%dxguyszFfL&*PWmKT!Y9$BPogD!5-oE45*)wC@mp~GBFAsR@rxY41;=lAUF{HF'
    '*M`^C4&ilecwOyKURRNl%j2XSK;iR&%M@-0491%gK?y`xyB=NV&$vG`o)27}8E-~NNeN1Vl$4+Ze9{%4z=-eHNBmR2J~h$=B}Gb_'
    'prlC2<t26t_)Kka{WJCa888?_7>o&D^)0V|)wiVAmBQ<yb$lzu2>3);$2VHXH(?#$XdT~#b$p|Bd=u93jn?r^SjV>_Kn?+rMSvUv'
    'Ad3Jw1V9eRLAG%{2iXSXAltZ}gKUEUIRrosV%HGZHHck9U>D`Pnvm~ml<#UnzN-yl*A&<_h+R`)*C2LHfn9^xH3fDJV%HSdHHlqQ'
    'VAmveO@UpMvTZ`jwo%Hq2`Sq~DcdHbY#XI)n~<_?lS26EF7_WzD{p`J?)6t+|L!>d^^b49d;Rj4ci+GM;g^^1UjKCd>A(L=?~iJ`'
    'j=Q}->izA=^`gaWGoI=F@&17``u_ESj~k5qN+azcBPAFiB`GNBq$C9;os^WIBuGgKO5(nf_j~P^jpRW_1SLsInxLdeNfVS5De0G$'
    'Jn`!@MoKV3N<vW5NJ$7v8YxLZNhc*KDCwl61SLUAN>BpqDuG=|>?(m>N$e_tT}kXJfn7=LDuG=^>?(m>MeHhpT}A9Ffn7!HDuG=^'
    '>?(m>MeHhpT}A9Ffn7!H8UnkD*fj)p6|rjw>?&f{5ZG13t|72%5W9xJu0iY?0=ovWYY6Nb#I7N*YY@AJz^*~;8Unipv1<tI8pN(C'
    'uxk*zrogU2?3x0*2C-`j>>9+bDX?o2yQaXdN$i>eyC$(~3hbK1t|_o<61%3ru1V~g0=p)$YYOa|#I7Z<YZAMbz^+N`S^~Q!v1<wJ'
    'n#8Upuxk;!mcXt>>{<f57O`sy>{`UGC9rD|yOzMNMeJGvyB4u)3G7<Lt}U=@5xcg)u1z7lg%I8p!dnR8O(DF65Z)BRTL|GzA-shU'
    '-VnlZ2qC;7gyT>`I7vwploTmxf|4R7m#5#SetpJBT{jYBqzohXUst;x!f{Xtx9cH%{)|RSE(0<e05aNjKt=;VM!OEkXcS#dh^|J_'
    ')rIJ46kT13t|TQ*P*S9%2}+8TTnA*Q=;}grb&9SKqN`JMg%DkxqAP^x>J(ieL|3Qi3L(0Jrr!{zUph+^!darAvqT}BB?>xA6vA1e'
    'ptD3FoFxi6OBBLcqM)-xA)F-&I!hG7S)!nW7$F?Q2-?gHVKXymGc$zE%%IK85H>S|HZw!m%naJh3}G`fXfrc}&CH<9%n&v+gLZ;K'
    '*a-^S2?}8+C}<}rgq@(ETzv?+`k-8W2)X*8Tzv?+`k-8W2)X*8Tzv?+`k-8W2)X*8Tzv?+`k-8W2)X*8Tzv?+`k-8W2)X*8Tzv?+'
    '`k-8W2)X*Fetl}B2}+8TG(ky`lI!f+#I7x{YZJS+z%JUP31O2aXp<&{O`4!hnh-W=f;MSF*rW;CqzPe@CTNo;g-x2EO_~%oX@WLs'
    'QrM&k+N4QglO|}BCWTF!q)nO>HfheE0X}KheUg-`Pa#*Il&eo6SD%!tPa#*Il&jCK_@qcl6O<Gwxjs%hjgu~n6WS0+VM8EkLm-6>'
    'fuxe$6iRZFN^(;u$xSNBO`#+=DUF_~_yn*bq|uYo=qaSplhWuZq|uYo=qaSplhWuZq|uYo=qaSplS*<^D9KH#0!*O_FsTYKg(|?L'
    'D!?q|bv06wf|5>3Qc%)K$z{|ReWsT1nOX<1tJ-yT)dB3PcAZ_-U@%s@J{YUPV61k1Fh2F`Ge!g@jg+LIq?3{qlyp*ZeK2<NNf&(5'
    '$tPX#NhhCl!6%)3(iNWsDJen8{cqtC)+a>=mP$CVRCHjegab=O2bM}WuvBzlse}VdMF*BjIIvXITC0TCT1Bn3N@%TB)LN^A)>=jD'
    '_<FjI|HEJZ?RS6q>g~IqpKpTJ=gsu2`mE{xzR_C$f7*MW9y^XC&-SYnw4Nry-OSuQYWWs#_jbK{je?NOKU17hOASd~g+Zgf`-___'
    'sY%}CJyndec#AtVfYeklEq`HV$KB1{!;X=Ey{n#mtS?Oe{y@ZM5B#^jf5J!$GO`3Cq$C9;os{HhC7pcI1)p?M(iJ70l+<Y@2W)OD'
    'i7$VPZ!@yMNC-+ADG5PI+*XobzRzzml0ilUB}qz}prlAi6O<Gw>8F(><w**8l9VSY<VjMVq>v{`<4FqRNfL!qpl}j}Q=ssbU*9ky'
    'C;@hrz^){AmB6kfc9p=cBzBd+t|WGqz^)>8mB6kdc9p=cB6gL)t|E4oz^)>8mB6kdc9p=cB6gL)uA)2{LY@rDlOg2Epgb8uo(#&9'
    'A>_%RJQ+fs49b%s<jJ5s8A6^6%9A1F$)GWQ3S<1BF@6eT{Gc&@3S;~rtV{tbldv)ctW3hn6tFT0D^tMAB&<vUE0eG?1*}ZM$~=#i'
    '=rB#a+y-8Pkp)Ih=Sfg0Duhy0P$?>eQdCeWDuhy0P_ZC{VnI-`AW~7%NJ$DxIw?s(Nhc-eGoF@nAuQ*DmUAI2=Yp1VAuQ)E=iF~9'
    'xg7nD^F}%uDZ$9?8RW%?@Vai#Adj=mo1o<O3^LpK8RW$XDQSX|A|t0q#-!mlh2b}YqAP^xqWzN;_D_=bPg2-FN!mY2VgDp)Ya@lN'
    'jijxO6t*_9$nisPJQWL4C>A6Y3sNW+Bozx%C>A8G<5O72C#~aCSjS(#@;8hKN{W=6Pev*hq);r#NseEF<0m<O367uS_$4@gk&<0d'
    'vcrV?cpwBLjf|Y03m46WOPC86&4o*t3l~jOOPHosgN(nNWj2f~FmgJ&icYGPa8m7Z9{RSDE`%^uiAtyv6%}wQ6(isip@35~B`RS`'
    'R5T?jVM<iAe^SE!NzuHkgn3udysLzHSJAwygn3udysLzHSJAwyZp^#xO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2M'
    'O_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF'
    '?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g2MO_T3U'
    'lkZKF?@g2MO_T3UlkZKF?@g2MO_T3UlkZKF?@g24rpdn<Y3=x?T1IUBD@NMCwBwU$`S*qC%;Rr-{!O*tA9!8BeEPt*87VN*w(~}c'
    'lr%v}k&=E|$?L)tjFe!6l!Tz9k&+OUG*Xg+l1@reP|`_B2}**Ll%OPTE2%GI{F{sbyGmeJ61z%ZR}#BQU{?~mN?=zJyGmeJ5xYuY'
    'R}s5PU{?{lN?=zJyGmeJ5xYuYR}s5PU{?{lhQO{Ob`61DMeG^^yNcK~1a=j%YY6Nb#I7N*YY@AJz^*~;8Unipv1<tI8pN(4uxk*z'
    'hQO{t>>2{Q2C-`j>>9+bDX?o0yQaXdLF}3Wy9Tjq3hbK1t|_o<61%3ru1V~g0=p)$YYOa|#I7l@YZAMrz^+N`ngY8fv1<wJn#8Up'
    'uxk>#mcXt_>{<f5Cb4S?>{`UGC9rD|yOzMNMeJGvyB4u)3G7<Lt|hQ*5xbVau0`xx0=pKmYZutHh+VtDu0`zH1$HfB*DkPY5xaJQ'
    'T|2RB7udBEyLN$HJF#mQ*tHY8c7a_xv1=FDwG+E`fn7VXYZutH1G{Py*tG+@Y7^MC1G{Py*tG+@Y7^MC1G{QF&#r5~zG0*UBcvn*'
    'C5@DXprnzK)9fm$3ztwAE~*QcP!}$$3ztwAE~*QcP!}$$3ztwAE~*QcP!}$$3ztwAE~*QcP!}#LMU_yBDk?>lP>L!lMb%VZS0^PY'
    'D1l@YiUmc*f)a`aMa6<TT`V}Zuy3D=IvzNED(cn9Eed}$Qi2gu(oZY7eJbjskveY#l$4;PNJ$AwZl95OeINuSjg*9-q>+*olyp*('
    'f|5>3N>CD{qy!~#TgmA&5^wqS4I`(O4D!hkd@{%<L-5I@WSl}WPD;ipB;%xHoI)~Aa{Lq=Kgsb^aXcjB5|VLJGA<z*Cryc#FeREa'
    'C0fFiXp!TW;P^$3-v!4na{Mkhev#vM!SOr1uC@uUYlqj>HsN*c@VeTjysjc8r}Lz3K;gEXr*PXK8QXR~8QVtD)y_v(YcMjlc79}R'
    'jeODspEUAG6MWLhCtdLgWPCrL@z;L6HqrzoMM|2Wq)5r>C3Xv#rnWdgO>IHR7(y~8%<5a7pVhad*OkKSqIG;L#R&LBSjRV7$2Vae'
    '-)J4*gmrwQb$k=n@r~B;O<2dbB0vrSkVSwT0w9Y3IRross6n=Iz6RL_)F9h9UxRFe067Fe4r133*foe<Ltq!xyP8n%YE<uPLcOaE'
    'V%HSdHHckPVAmjaO@Up5*fj-q4Pw_6*foh=Q()I5c1?j@RI+VC$+l6+wh1NMMkU)Olx!Q7Y@1NBZIeRydKddohn1I~et7!skAG;#'
    'zyA68ho|rV`r*f?pZ@y&ho@hTKmE5)zWtHM|N6~uf8^^gk?)Q_J05-G+aG!PCEEVk1D}6j<R3M%gOQ$RjT~iRQqp(92q_6cNh2j8'
    'C~2f51tp!7q@bjek`j~zDJelo+*Y!G8ROq%B$JE?N|KZ`K}nI4CMYRVa-Ll~v1=FDwG+E`fn7VXYZutH6T5bST|2RB7udBEyLN$H'
    'JF#mQ*tG+@`X;bz2X<YI2uhNaG(ky`k|roAQgWJI*M5D&$SxQmB_Swjq$C6-jg+LIq?3{qlyp*3f|4L5B`5)Qwe#${_Uju)1SLsI'
    'nxLdeNfVS5DLK!sM(k<=yBe{p3G8abt|qXn5xbheu14%?0=pVvr3qMRgq1E}r4d%TfR#pA=>k?7VWkULX@r$7V5JjQx`35VSm^>*'
    'I$@;?Sm}h7E?}h-R=R+dPE(>TOo=+Ns|)Pv#I6w7)rnmpu&WchLSR=Xc7?#MPV5SST|prnLI?+ia0nqB6v8Qla8L-R5W+zroI(f('
    'g>VWX92CMSgmBXEo5Jv$H2kJ8{3Z>*DGa|!!*2@1Z_@Ca!tk5Kt`yjn#I6+BmBg+R*p<Yt64;f*t`gXl#I6$9mBg+R*j2=?64+J5'
    't`gW)#I6$9Rm83m*j2=?64+J5t`gW)#I6$9Rm83#u&an&LtxjS5FSDZ4+`NSgz%sc9zqBY3gIDy@SqSLLI@8E;VFdhpb(x)2q!6N'
    'f|4R7O;A##<oxhEY51MO@H=Vvox<=tiCt4**CcjLfnAf>H3fFj>A^0X9_(~_unVULJDncv!s)>XaD2ph$47wUBhEWMXdNHIIzDI}'
    'AHq65XdNHIIzDI}AHq65Xa*U=3^Hg28Nv)Q3Wnbp=ZD`Y7=B}%AAX}?_>FOX_`UY)8%B1)2q_6cNh2j8C~2f51tp!7q@bjek`j~z'
    'DJek-uuE7v1T7sxSULnP9YR<-1T7sxSULnP9YR<-1T7sxSULnP9YR<-1T7sxSULnP9YR<-1T7sxSUOz$_4<Jllmsa$K?$&H3GAB0'
    't|hQ*61$ebF1kAx!ri%`yK^DjoeNt1hOqh#TK$Hw`VCtBhOzpki$@_`JPNva6vD-$po>Q#Ts#W8cof3Lqo9jNAzVBPx_A`A#iOLd'
    '`zajWPddDx!r}d-!}}>5-cLHbpTgn&q{I6u9Ntelyr071{iMVDDIDHUI=r94;r*n;`zajWPddDx!r}d-!}}>5-cLHbpTgn&q{I6u'
    '9Nxe7>-7UAC<#(hf)Zeta6=&JhCm881d?tDq;NwZ>4rcGHw2Py2&8aBAnArc3O59jZV04sLm=sfKngbml5Pm3a6=&JhCm881d?tD'
    'q;NwZ>4rcGHw2Py2&8aBAnArc3O59jZV04sLm=sfKngbml5Pm3a6{nQuWuOH1tX*+1SO4>grKC6k`$D5Qj&s_PD)Bp5~QRACBQD>'
    'YDm(_f)q{`B%LhC>yrh4`rH5b!+(AE^5NIFKONSO@0V4ZRbMsTUq4TFx!1hjRnI=w7p8xIAmXzJ{#)NaVWb5aS%MK#l7f;>O7gUl'
    'PCn^^PdX{-ijq!B>a>z$l)0@WzWgn|&By{HAt-62Bm^aKTS<QTKEKIG1{o2QBq?cvk|HHdP*S9%pH`BTCn@AfQl6xcCrNpdLY^dz'
    'Cn=04Nfb_j!bucPfx=gQeZz>L1lUyqyOP*d0=tsfRRX(`*i{0%lGs%OyNcLV0=tUXRRX(;*i{0%ir7^GyNcLV0=tUXRRX(;*i{0%'
    'it=O#c`_(ZhL9(N@?;2kGAK`mkSBxkWC(dOC{KovCxh~22zfFnPlk{ugU0wNjPZlU_$iF>gU0wNjPZl8G6k$m!pan|G6^eFz{(`7'
    'OaUvCurdX#Ov1_(urdiN^E_6f!!-4BQEdrE78p65CqbpC5K2)&rKk`}Q9-4s5K2)&#exuu1wqAvNJU8_B`GNBq$C9;os^u<cv{Yd'
    'u$&88&V{g?3tG;Fu$((i;@ys}%b~+KZ={ov5{%rQL0*gquj}><@^T(fP;z?)neF@x@?wOPG(ky`k<%k%((s$Y@S8!=6+(2;{z(e^'
    'CrSG!DeRvl?VqHuf0DGdk;2wS($+=_TN_#A_#rr+iUlbY3zCWjDHIEmiUlbY3zF9HDXim@*6}H<<F8-&8%6{rMM};mBNYo$C>G=-'
    '$1lP0lN`SU$4_$n5*)us$u20_VZwbp5Q33LMo!O#i{`>5%!P~Q!X?awi>9e1OjD~t#vc!K!^i?7r=zRrq*@6l)v7}X9}k=lVX6|9'
    'P$eoV;8ZF`z$Zchr)Wx4!j!0JN>sv>sA&JBg#DAEc~=SZuA+HY3G=R^c~=SZuA+HY3G=R^c~{+-cimeJ-&+meTMge^4c}W0-&+me'
    'TMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0'
    '-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^'
    '4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMge^4c}W0-&+meTMhsJqt)<dMyl<#-8AyCS!%s&G24tcjC|}tx_$P*HyLRg'
    '87aXCDM>*|CnYH;>7=9tB|%C`P!hM59El_-$&HK%N|KZ`K}nI4CMYRV(oZY7^6MK$N-#o7LQv93NeD_BDM>*|CnYH;>7=9tB|%C`'
    'Py*~Kfn7=LDuG=|>?(m>N$e_tT}kXJfn7!HDuG=^>?(m>MeHhpT}A9Ffn7!HDuG=^>?(m>MeHhpT}A8~0=tUXH3W7Qv1<tIDq`0V'
    '*j2=?A+T!@yN1B7LF^g=y9Tjq2<#fft|72%5W9xJu0iY?0=ovWYY6Nb#I7l@YY@Apz^*~;ngY89v1<zK8pN(Cuxk>#rogUA?3x0*'
    'Cb4S@?3%=`DX?o2yQaXdN$i>eyC$(~3hbK1t|hQ*61$ebu1V}#0=p)$YYFU{#I7Z<YZ1Gaz^+B?S^~Qkv1<wJTEwm;uxk;!mcXt>'
    '>{<f57O`sy>{`UGU0~NDcI^VYb_(HL2;rSVco#x=rx4zS5Z)<-cOis#3gKM{;T=LaHX(#}2;qwnK}nL5CMYRV(gY<%N=^^I*M5D&'
    'NS!wlWTXruw_jH~AHs(b2;p`<gpZ%mNXcnH9zUa#k`$D5Qc{AFASERzf#~W&bTx{uu0&Uok|roAQqlw^MM};CvQu<*A-Xz6R|wJ7'
    'DY`<4u1?VvLUeVCt`MTDQ*?z8T|vWd2*WR(B?{pzQP5eU5Y7?>oh1t4EK$%|q7cp!1)U`d;Ve<mS)vfm5(S+l3gIkK&_Rq44q^mt'
    'W`?kt8MK)h!e(aBW@ZSRnL(SGA#7#_ZDxkBnHjX18Ny~}&}L=`o0&m7K_Tn}1?>cduoD!t6BNQuP*ANtgj#)2tv-ZWeNe4Fgj#)2'
    'tv-ZWeNe4Fgj#)2tv-ZWeNe4Fgj#)2tv-ZWeNe4Fgj#)2tv-ZWeNe4Fgj#)2tv-ZW{k31OjWj_?k&-4TDN=HtT|2RB7udBEyLN$H'
    'v`G`fCQZ;LO$eJbL7OxoY|;d6(uA-{6SPSa!X{17CQS;PG(np*DQwaNZPKK$NfWe5lfouV&?Zd^n>0zAG%0M-Tt2jOTS+_blcZXG'
    '3bp#AT73$&`lMQY3bp#AT77oKCq+t{prlC2`8?^ACtb)B+7L)#Lm+8GAcYNqq${~8T**zklAFSn+@veHDO|}-Dx+sAJ^`!<W%Q&n'
    'dJ1Lqq%wL6W%Q&ndJ1Lqq%wL6W%Q&ndJ1Lqq${~8T**zk1(?Duz@%G%Dck}~x&@e}yskz{Qc%)KNeW6jDLIY$qG@Ug)6~PrZFW^V'
    '&#u~lUDeLBs~RL@we!hX4U)0i`DDEI>l;P{C5@D%prn(M6qIyQay}V5`J@Xz>Ex3x_@t9hy5N&eKIw{2f|Qh?<n~;+g!M_$fu#}-'
    'EEOGCD&fFV(SfBB4lETNSSsPbQqh5>5)LdCeOjx8Piqx@TC0RlYZZN3tAtN$6|LjzdL93#!^+D~KRkW+$3OJrU;q64!_)VF{qW<{'
    'Pk;UX!_zOvpZ?p&U!t{nd~+>g``S0x+P}2<+qd+#cQcQ_u|E4Het+O~f%EADzbpNxj1(AY+j%2JN}8agNJ&4f<aOZ+MoKV3N<vW5'
    'NJ$7v8YxLZNhc*KDCwl61SLUAN>CEFmDHCp{!K=JT_vz9iCrbID~Vktuq%mOC9tcAT_v!qh+QSHtB74Cu&an&C9tcAT_v!qh+QSH'
    'tB74Cu&an&Lts}CyN1B7B6badT}A8~0=tUXH3W7IV%HGZHHck9VAmja4S`*Q*fj)p4Pw_2*foe<Ltxh+b`61DgV;3%b`4_J6xcP0'
    'T~lD!Aa+fGU4z&)1$Iqh*A&<_iCt4**CcjLfnAf>H3fD}V%HSdHHlqQVAmveO@UpL*tG<9O=8y)*foh=OJLU|b}fNjli0Ncb}eGp'
    '64<qfT}xosB6cl-U5nVY1a>WA*Am#Zh+Ruy*CKW;fnAH(wF~T8#I9Xn*CKZ90=pKmYZutHh+VtDuASJm3+&p7UAw@po!GSt?AnQ4'
    'yTGoU*tHAn+KFAez^<LxwF~UpfnBu;?An1{wF&InfnBu;?An1{wF&InfnBwoXV<k~-!M{w5mFL@l155GP|`@rX?7LWg-fUl7uAJJ'
    's0$a>g-fUl7uAJJs0$a>g-fUl7uAJJs0$a>g-fUl7uAJJs0$aBqDm-56_uh&C`A>OqG~FytCNxxlt3~H#e$+@K?%izqGCavE*2bH'
    '*tbtb9S@v774>T57KJ|=DZvOS>8F+4J{9%RNS!wVN=i^tq@)BTx6eqtJ`jSEMoL0Z(nv`PN;)Y?K}jbiB`66}Qi77Wt>p9>iMRav'
    'hLO`s2Ki(NJ{jbbA^2ocGEN~GCne()l5tWpP9Yg5IerR`pXB(dI3AL53CTDq8JCcZlcq#Vm=aBz5-nj$w8-&GaQq_2?}FnOIer%$'
    'zsT{s;P@S0SKEZwwZrRboAA1JcwKE%URRNl(|OW1pm5vHQ@CxAjBPugjBTUnYUiV?H5eINJ3lhEMm}kRPa6582|j7$ldkv#GQOYB'
    '_-nsj8)<@)A|*{wQl#Yc61xRVQ(K&$rnaDD3?Ug4X7w%4&+1#!>q_Bu(K^1BVg!65tm7N4<D0OKZ?ukY!aBauI=%_(_(tpaCamLI'
    '5g>;E$Ra=v0gy$290DK*)F9h9UxRD|YLIQ5uR*p!fE)rK2eE4i>>9+bA+U?;T}`NWHL7<tq2AR7v1<zK8pN(Cuxk*zrogU2?3x0*'
    '2C-`j?3%=`DX?o2yQaV{D%m!nWZS4@+k}#BqmpeCO16zkwoNG6wn-s;y^H;)zx|It{MUCcAAWuFV;rwv1j*xn{pJ@z^3`;{JO1o='
    '^o=iq<i&K`KYQTw&yM_~Ms_gL^Q@7h7)(m~E*K#tAt-62Bm^aml%$}fladsabW&1+k{~4|D2dxj_Ag`nn~Y?V5kX0kk|roAQqlw^'
    'MM}=IYbSQ?0=sr%*DkPYCwA=uyLMvNF0gAScI^VYc4F5quxlrF?E<@YU{~J+cJ08fixELdl9DDUDN@n|B}Gb3v+LTgZy4DHBcvn*'
    'C5@DXprnzK6qIyQl7f;>N=i@?q@)BTz^-<lUDtkn!-$|HNl6ow6e($fk|HJN+0}?$O<-3eb~S-rjo8%$b~R#G6WG;=T}@zDBdjz5'
    'D~+(y1*|l}N*A!w2rFH{N+Yav0V|EL(gmz^!b%sg(g`bFz)B~qbO9@!u+jyrbizs(u+nKt)P*TgCw6s#U7gq!0=qh~D+G3RVpj<4'
    '>cp-P*wu+$A+RebghL47pb!orgo8pjg%A!3;S@qRD1=i8;h+#sA%ufMIE4^S8h%q4ev^jZ6o%iV;WvfhH);4yVfalNep48Jlh~C4'
    'yOP+I0=tsfRRX(`*i{0%lGs%OyOP*d0=tsfRRX(;*i{0%ir7^GyNcLV0=tUXRRX(;*i{0%ir7^GyNcLV0=tUXH3W7Qv1<tI8Wh4q'
    '2;o5?JcJM)6v9IY;Xxrhgb*GS!b1q*K_NVa5FQl5QwiZDB~4ILq@)Q-ij<rmekTpTQy6|H4Zl+uekZYO3hbK1t|_o<61%3rE;>Ef'
    'h0}wbP7ikB^kApcgIzd17y*utIPdrfaD2ph#|N$BLs-WLt>Z&j#|N$BLs-WLt>Z&j#|O<ILzqDZ%^*XVK}Nyw8{_=&8wJB}jPt{9'
    '6b!#H&JVxWetpBpE*K#tAt-62Bm^aml%$}fladsabW&1+k{~4|C;@f}ONXGPLkLTUpru0yONXGPLkLTUpru0yONXGPLkLTUpru0y'
    'ONXGPLkLTUpru0yONXGPLkLTUYrkGUP=b;mB_${Ub}fNjli0Ncc1>c}64*s|=R&wU7j$<ngu8P=tKSe-zd@_t5LUlItKTqIzjW~^'
    'go{T(7mq@?cocN;D1?hgK^KogxOfzF@hF6gM?n{lLb!O8ba+37!~02x_ft5$pLBRXg~R(vhxb!Byq|P<KZV2lNr(4SIJ}>9ct3^1'
    '`$>oQQ#ibzba+37!~02x_ft5$pLBRXg~R(vhxb!Byq|P<KZV2l*M7ZzpadmBN=i@y>=JGWB;625;f6rc4S^JH2qfJQNa2P+(hY$W'
    'ZU`jZ5J=&MK++9?6mAG4-4ICOhCtE{ffQ~CB;625;f6rc4S^JH2qfJQNa2P+(hY$WZU`jZ5J=&MK++9?6mAG4-4ICOhCtE{ffQ~C'
    'T>JG6BfDUPl!Tz9k&+OUG*Xg+l1@reP|`_B2}**Ll%NFIC0q?jI$4my$%3Sl1qmk$s_nJiG*YkEu<Bil*=D?9<n@8$!*riM@J&YA'
    'Mn+07LP}Cl(n(1QN;)YiK}nF35|qSkC9iF`P8-RMj0j4Slr%v}k&-4TDN@o;E4lLP8%9bnLP|nV(nv`NN*XCiK}jbiDJbcrqy!~F'
    'N=i@y>?(m>N$e_tT}kXJfn7=LDuG=|>?(m>MeHhpT}A9Ffn7!HDuG=^>?(m>MeHhpT}A9Ffn7!HDuG=^>>2{Qir6&-b``N}2<$3i'
    '*AUoM#I7N*YY@AJz^*~;8Unipv1<tI8pN(4uxk*zhQO{t>>2{Q2C-`h>>9+bDX?o0yQaXdLF}3Wy9Tjq3hWxht|_o<61%3ru1V~g'
    '0=p)$YYOa|#I7l@YZAMrz^+N`ngY8fv1<zKn#8Upuxk>#mcXt_>{<f5Cb4S??3%=`C9rD|yOzMNMeJGvyB4u)3G7<Lt|hQ*5xbVa'
    'u0`xx0=pKmYYFUH#I9Xn*CKZ90=sq!;av#fokDmQLU^YT-h~j}DTH?+gm((zT?pYFLO3=dgm(zx*pv`XQqlw^MM|2Wq)5r>;rH6F'
    'Zy2fbMuLo#VdVDfYUe}vFajam&WG^vGa4y54ajH!$Y|#Q84Um#?K~i(QFJvSx*A1S7ow|Cbaf@Vl9V(-Ns*E!C@E5M9*~`)s|(T9'
    'DY`<4u1?VvLUeVCt`MTDQ*?z8U7eyUgy;$yenS|3=`2wQXNiK&5`}P<DCjIv2xp0c&Ju-imMG{fQ3z*=g3c0!aF!_OEKvw&iGmJd'
    'gm4feXfrc}&CH<9%n&v+gElim*vt&t%nV^OGiWn2gw4#L&CC!sGlMoWL)gp=+6f9_Cn#tqD1@D$pq-!)c7lRx^&!;igKG65)arw3'
    '^&!;igKG65)arw3^&!;igKG65)arw3^&!;igKG65)arw3^&!;igKG65)arw3^&!;igKG65)atMOdTpc$N{W;;K}nI4^X%G*UAw@p'
    'o!GSt?4nJY5H@LoHfciGqzT%j31O2aXp<&{O`4!hnh-W=f;MSV*rW;Cq)B0uCTNo;g-x2EO_~%oX@WLsQrM(P+N4QgljicFo!d&<'
    'd7mWJ>QkuIC)MgxsMROc>QkuIC)Mh+D?TYw(gY<%O3vp=r#$IGp3sIs3L64R8v-e82qay}P2oyz(v{p4uH+_N$xY!(Zc-UNQ}GF4'
    'MJS^umC;iuqbHToQz)Y+mC;iuqbHToQz)Y+mC;iuqbFU-P2oyz(k;LgZUH9U0!-l+VA3tXEai1IQj&s_PD)Zx(n-l_)E7-tOPHn}'
    'MsBmK+Ie=>2JEVKo?X=-8LOR7#%hp^)y^m5wO`*bA}DF3Bn2g%l%$}flalkv*vThd@JT11bipT`e9{G<bn;18d=jLj1SPlU!X>Ov'
    'iViH5aA2wEz)}eZmWmE6m2hCG=)h752bPKsER}Fzsp!*MC45?|=+jyyd|IpM(^@5bTB~RsUteFxw>FP&szq#H`=(m^mo|U<R^Ik*'
    '=J7YyXGVU1;B^7>=>y+pq`*kq&KoIG(gY<%O8RLfuM1N!Qi2gu5`vOON<vW5NJ$DxIw?s(Nhc*GC<#(hf|9tcq`r*tZ!!YxDuG=|'
    '>?(m>N$e_tT}kXJfn7!HDuG=^>?(m>MeHhpT}A9Ffn7!HDuG=^>?(m>MeHhpT}A8~0=tUXH3W7Qv1<tIDq`0V*j2=?A+T!@yN1B7'
    'LF^g=y9Tjq2<#fft|72%5W9xJu0iY?0=ovWYY6Nb#I7l@YY@Apz^*~;ngY89v1<zK8pN(Cuxk>#rogUA?3x0*Cb4S@?3%=`DX?o2'
    'yQaXdN$i>eyC$(~3hbK1t|hQ*61$ebu1V}#0=p)$YYFU{#I7Z<YZ1Gaz^+B?S^~Qkv1<wJTEwm;uxk;!mcXt>>{<f57O`sy>{`UG'
    'U0~NDcI^VY7O`s=*tLjVyTGnR?Air(?ZmEKVAoFU+68v)#I9Xn*G}x(1$OPku3cc)PVCwRcJ0KjU0~M^?5a&**ADEeO<>m!?5a&*'
    '*ADEeO<>m!?5gcNyRQBEhLIACkdhFTG*S|Rl155Sv#Y2sTtZ#As4iSWUAU+&TtZ#As4iSWUAU+&TtZ#As4iSWUAU+&TtZ#As4iSW'
    'UAU+eRYEDMs1#K~DXORxRa1Fgos^`Y1d>rG78Df=N+=c-6$|QgvEbOkzI`g{c;NJ@s8=JmDE!e#2}VdsKdt2Usi==e>bwz9Qi75q'
    'B_$}ieMaK-fe@55QWAoaMoLmp(n(1QN;)YiK}nF35|qSkC8y6wyye$7jGR_7$R|Va$snH$!6%cFaSF*eDH*4bjFXaa3duOh@l$a8'
    'B*#z1@sNy5NXALYxP)Y!G$mTXlxWhFXbDrIMUG#B;}<!87aYIH@w?#oMULME$M5jE+9tfN9bQ-4gx9sh>uQ_wx{8#X&XcwQh1+(X'
    '!fk_OY}@%{Y#T*aJ0D%G!N}O!`H`_T@<|hX(#R)G@JS<|bj2r-@%?<pU;FjiNE4J4DQSX|A|<Dn*ezh1+T#2)wFM<(2+5c*t8aOJ'
    'R^O6dR|>C-*72<rBj6KZ9p7jj--LC1qjh`}*71$j@l9CAH(JLxVIAL!067Fe76EbyfGh&!5CA!#2HD2>8e|(#gKXn`4YCaa<PZQk'
    'h+RWq*C2Kcfn8MZYC^rMQN61P^{zIET~lD!Aa+fGU4z&)1$GT$*A&<_h+R`)*CcjLfnAf>H3fE2$+ih4+eRhZCX{R&m28_(vTand'
    'Z9>VmO$y=bUF<&{R$hMk;pw|S{vnQk{qyq=Pv8Ib!;eot{q_3~Prn?0`fq>#_D7$WZ-30HuYUWZzuM?Wtzf;Yo_(w@zWwq00}-D+'
    '@ZTR8`6rCDAR|jKLP}Cl(n(34R?^8QUGPaKC0$X{NlBena(wOQwvzbrxA-<A3yg%Iq>+*kl*DZ%`Q`ikCL<YSL{O5XqzOujlr%v}'
    'k&=E|Nm8DqkS9rbl0u#&<w**8k~E&AFrFk)I0XtPQ8)z(U-|V7BZ3lOR|)J&Vpj?5N@7<D>`G!+3G7N@R|)JYVpj?5Dq>d&>?&ec'
    '3G6CjR|)JYVpj?5Dq>d&>?&ec3G6D$lOg2Epgb8uo(#&9A>_%RJQ+fs49b%s<jJ5s8A6^6%9A1F$)G$LLY@p7<EJpj4;tgAFvbrW'
    '<EJpj55me6urdiNQ^3k3tV{tbldv)ctW3hn6tFT0D^tMAB&^KyScwkP)T7z?>1pc4$O0p$^CYMg6+$U0s1y}KDJrNG6+$U0s8|p}'
    'u^^~e5UD6>q$C9;os^`Yq?3~K8Bfc(5SDX6%efGib3x0w5SDY7Mw2&{TzW6Yc_W>Slwjoc4Dw<`cwM(=kViAYO;B=s2AS>r4Dw=x'
    'lr%v}k&)9QW76=O!tk3x(G@~;(f&ya`zJ~JCn@ZoB<-K1uz!-YwUNTsM$*<s3R@dl<oF>to{9x26bq7y1t}B@l8OZ>6bq8p@hPn1'
    'lh*MmtmCg=`5Q(AB}GckCnFULQYaSWB*!no@sk|C1jkQu{1P0$NXafJ*<r$c`2d(;q>+)+bK#=7a0zqaqPcJhbK#<CY6;WSYLM}l'
    'W`c&11x8LsSJ6qe5>BdJ8WP@C(uEMFDp3hlqM`y$rD6npA{20nrbH!7iHfE~B}|Em_D@RKKPj4bl`!urns=2j?<$&il`!urns=2j'
    '?<$&i)%CpVPk;N<-ws3n@$mDnA09va$HR}memq3{e?5Ho_0JDKUw(=I`{DhEhaWHh_W%6!`j7wp@V_5k4<i5a(|`W_`279l@4h@d'
    '{`&nde}3pSj=%q3-@SkM<@tvXKR$o|^YhE|&(A+TKL7IWfBka&{XhQUyN8$Y^0@VA?P2cE@sP2#m5<|nkB9AXu2uE$5Y_ur+h4}}'
    'ZF|^W#>nmYVU2ozc)t7&kAM08hv$c<=bzvI^zif3yT@Pta{0;m66^ir<KFhIZ!hcN>7hT&$LD%{p7qjS9-pG#uY7nu{<&B5wr%^<'
    ')82EwWP8~6e)-9N{`oK8KmVWS?=PpUzyJ4#A3wZ2|MKhazx&JM>}wm3`*@7^t-T!o_xtVr^Gm$+$M^kV<jc!Tt+vMFoU=c)?R`7!'
    'KlgsvtH+19Son`09)A4&2R^?%ws?;IoG&fjA3x%yYL1tY4>O)#+H+jq&-3c<+syZeyx#KoowwshXS{6N>#zLFfBf*_*Y6*G`Q^j^'
    '{PEwP4;TLSKH@Q>Jv~0w%S+#$p4!X9<u$*rwjSooirlw)tbN9OT2aq0PseZGp7-OG&(ZswuW$M<4?lc(dieFjPd|SD@6W&f{NeHU'
    'pZ+vnV#^$DjEASSKR>kRL&L+%Lu7kAeD?nMlgD~qFYjL-pWh!2*&pg9$M%xjjQ3Gr{Ph3$`T6O?<K<s}w=f@{*83IjA0OKBk{(u$'
    '{i(mK*AUsCpGLkPwf9<wPuHw?*dx{vZ;!_t8tbLycYpiuD1H9Rho?&%zJ@0x*5@Jo_|spWe}4F%m%q8aJKoTqW41o}TwB`@1HV7`'
    '^q)U}l=92-<4-@n9>;G7z_#PT-~M05r+?x3$EWXqe0^;0v8}yFe9X?{$xomA`QLx}<p{>hlRe+{%$m`wR?TgF`rI$SKK%Nhm#4n_'
    '`AClcb9uopAAbJj*H=-WS&D71Rv&+-&-4GdM{xN8uStG+d(TH$9g;5T@ci><|K;O*{Uqs+)cMVG&-wm&?72VIdT#N!j!O)4e|niO'
    'J)fT*U!ISDd4FWf`|1xz_O<6DLEGatx5|3|{P^@5)~~<x^Itv-=!eJWr<d5bOF%y!k3YUdJv~Jn$=2t?<9K+E`Owzm7%xXatRq{W'
    'pX>cgTe0Sm`rrNSZ=ZEX&EtRlpYZ-(s~>ND%v}5NPxStAi6qB?gM59_E%K}0k0X&{KYp|07y7#QkAeM&sO^AZtKLRi^$qU7n3^;D'
    'y5au+hW>Q`eDdFJhQRZY<A;+D0z9sK8jnxg`}MNd!}Ie#SN4~g+kWKHOKtl>(W6i85t7m7!~1x8%ID{=gus{wG9P=4c>lC-k5BKP'
    'UY_=2czBNX(k^#p`Vj`t$A6ex*30|%^P!&iHJ;Wn%slPS?aL5&vqA2y5QvNG*NXjs<Pp^v0tWytVckE5!0b0dU{zkga&+1KdI%ho'
    'mHi_D4*SPIw|`p*tkL#9$Bhv9WD0N{1E2oKTS1V=xbr+7w-t}iFUKG^dwV%XntVQn_Bj6J81ZHtNqivTz{C5T(bj(O;25^s`)n^?'
    '34;B7uRKzE#lhD8@P5`qt7Bw)csa)Q?DH7t-oG3}<?}J{yni|}zprCV-D9i^WVVM-g5c}dwQdE$tZ{gu=NOgy<RFN5+n6JN8wkD{'
    '1jmrv+Soqc`TBYgwDE4+F9|dA7>Dx6hJQW?UQJ!%r`-sGzfbwD1K_v+aLf5^f4|4m?6V!bKc*G+IQRE2?+?EpW85}cReO5aW^I>1'
    '91r6q*OB`@_hZ<2dEVZC#rY3U$JFTY7=oW4A4kqSf@hD1_x+dxtm91|1H^%cwm(14$K(G;dtB|9dK`bHy+4M{whrfiGB5b~KVRoz'
    'zrVbXLDA!1+hsUj|L^zfX+0i;$VVO>bHT6Nw;Puzn%n&PBz^q7uk)7g=7H{LUry40d!X;1J@Cm&;GZzEgOSMdMj9yzK}jPeDJbcr'
    'Bn2g%l$4+(NJ$Aw;<l3fGRD8jNG2H(lq4x>f|4R7O;A##q@Pxj#I6+BmBg+T*p<Yt6xfx-t`yjn#I6+BmBg+T*p(E*C4_KM2$vAT'
    'MIl^52p5HL2_ak*!X<=oQ3#h1!bKrmLI@WPza<R6MZ@n9hTo#$cL>98(eOKj;kRh`9m4Qi#I7N*YY@AJz^*~;8Unipv1<tI8pN(4'
    'uxk*zhQO{t>>2{Q2C-`h>>9+bDX?o0yQaXdLF}3Wy9Tjq3hWxhu6cu9m*QEk9yj+U^K$zp+EKgS_c7ntn|yts#%B+Fn~`zT$Wa$*'
    '=Z$0|BZ877B~4ILq@)Q-ij?%zO0N9+hLIr{AtfOwX{00sC5@D%prn(M6qIyQQi75oB_${UcBR0sAa<p|t{`@$z^)*6rNFKrcBR0s'
    'BzC31t|WG)z^){ArNFKvcBR0sBzC31t|WG)z^){ArNFKvc9p=cBzBd+uA+<|LdFlu_#tHcpo||v#t+K)A!PiZj2}YA56bu<Wc;9v'
    'AGb69ANpd$q7Pgw;R6?oK5(&w4_qwzz{L_iaIxqE7fbkV&Z6(;EaAI3i@uw)3*XII^xd3Y_-+neR_elKrR$$@{lNJ=q035LxU3W#'
    'edk~kzH_kAcMdk;D;o5jgV>bU)k#STN;)YiK}nF35|rG=N(diQ2)d;f!Y#F+k12%kF@<ZtzF|aAQl#Ykw~QJ^R}-SEQFL`7x*A1S'
    '7ov;4R~W+g3WL607Sl1Zz{q(D)7Q&F_<C8;*UQ5AdfB8rnL?gS%9ADJ30-#&;ktX!b@vdiyI;TZYa{0YNwfM8X7xeS)DWhrLDSSQ'
    'rl~t+{4QktP8q-7$oTbr+m0&=<LgG&zg{}HzV=2P$E}k#zBKaN14j#*PapU;BLzlkJ8vXNNeN2gwvyN0sHctOMn(iBNlKcaq)15<'
    'loToHr<Gj!^$jB>7$GGgC~2f51SO4>q@bjek`$D5Qc{AFASERz0d|GJu1@R<fnA;06#~0Du`2|2bz)Zt><VI62<!@CR|xD1Vpj<4'
    '3Sw6X><VI62<!@CR|xD1Vpj<43Sw6Z><VI63hWAER|@P3!b%EQNy17BSV_W43Rp?PN(xv>!b%EQNy17BSV<Zg>+HyQl*n(7jF&?F'
    '*${sH8Mi|CqmdGfkdhFTG*S|Rl155WP|`_B3Q9UDDM3k)k`k1}Z6&8i#<%?Xh7mzYl9F~>Ns{AJaD0;EQ*eBe<4bUSlH*Hoe3Ii!'
    'aeR@ICMYRVa(>V%8nj9nw2B6;5(cfJL92v8t7y<FVbCfXv`QGXir7^GyNcLV0=ovWYYOZd#I7l@YY@Apz^*~;n&;Wo7O<=B=h=1q'
    'jN9zG7%9OBDG5PIBPAgyX{00tC7qO{prn(M5|jifDM86?cJ=K%yN;i6+b72Zr+soULP|<d5~QRACE$~$_~iC@axo$(Nm9}TB}Gb_'
    'prlC2>G9;+uWuMB!3Ze{K}jPeAt-62Bn2g%l%$}fladmY09Lwyl}1?U0#-UXz6*};<oGT)zLVp-;P_617VO6#-(oH@^0B$Wc6@}r'
    '<(o!6egyy12foQj1{qm`5mM3wB}Gb_prlAiKdt1-uWuMxf)P>@f|5o`LQn$HwS?%J6kSV*u1V3ggy>onT}z0rMbWi{=vowAONg#3'
    'zg|C(f|5>3Qc%)KNeN1Vl$4+ZM#fzj85fO=yD%~?V%L6_U6(27O?Dj*w6pAb{Ta8E^vj0*c_sa3GQK{Lf{`9%WC=z{NeN1Vl$4+('
    'ZYzni$@rFE-!LL5Nm62jaFLQGC@E6XPb-O=6N!&TmSBXGgrKC6k`R<MQj&s_PD)Zx(n(1PN`jQ!WY^K=B<4*cap}~4xu>-q-8t49'
    'MqVGd+|&Bxfp0Uiz(~bOBlWT8{MU_Cd@1!CjJ#$%7+Hc5QWAoaMoL0Z(nv`PN;)Yy9m0R_eR{)43O-5lNeVtm@<|FlN%Bc5KIx>S'
    '1SLUA>a>!g*Hyyn8uYq`@VW-Qt|7dx`DGe>`M~-T)8AwygNz7Dl9V(-Ns*E!C@E5Me)wH9{4QbmU4;5|7WJ0{$2U>``oKxle>Aed'
    'ND4|iDY?n6qYL8kkK0E2ynI@4&Dmyq)5!4^vEws~pFZ$SMzWESAs8VgO;A##qzOujl=Ra|uKfCjks%l%B_Swjq$C6-jg+LIq?3{q'
    'lyp*3f|4L5by~?zj^72x@8tMhaQqJJx_r`4Fmjt+7bC*!DpGQqU6)Tal9JQxx)>oPAt<>${9eA;Cm6Xs{9cR*uPaGO6O<GwX@Zg>'
    'CFeu9Q3y97gd2r$6GFHFg)c?~C5@D%prn(M6qIyQQi75oCAWv)qa}9UG;&;R8m|Y@X7xAA=*I)|^-$g?4}61>kGFAd8!5pEDG5PI'
    'BPAgyX{00tC7qO{prn(M5|jifDM3lxR<gc~@ozGc8yOLlBq?cvk|HHdP*SAiJi8XLYYFUH#I7Z<YZ1Gaz^+B?S^~Qkv1<wJTEwm;'
    'uxk;!mcXt>?Air(En?R$uxk;!c7a`s*tHAnTEwnhVAoFU+68v)#I9Xn*G}x(1$OPku3cc)PVCwRcJ0KjU0~Nv?Air(?ZB?T-<CaV'
    '<Tkr5Mg%2EN}8agNXcn-UBB|{2fAT|Ue{@?T)&O$2U0N7>2;;>x;iN-K}nF35|ltPw)4q&{mQS6G~sm>DKV0<NXhAm#NV$;-!@W$'
    '5mFL@l155GP|`?A3Q9UDNkK^`B_${cQc{8vm`L>V6NziTzF|aAlBA>wN{W;;K}nI4^X%%xt}d{v6T7;=u1@Ug0=qh~s|)Pv#I7!|'
    's}sArz^+d0>H@nuu`2|2bz)Zt?CQj>5ZKj;T_Lcm6T3oSR}i~GU{?^kLSR=AyFy@B5W7NPR}i~GU{?^kLSR=AyFy@B5W7-fR}i~W'
    'U{?^kQeamQyHa3R5W7-fR}#BYU{?~mQeamSyHa3R61!4hR}#BYU{?~mQeamSyYe=>Dx#5*k5}We+F0{VBOkA%efq#R7%4I`6eAU6'
    'WC=z{N&lLX*p5$YwH9AL6&2g1H$+?-g}<xlxqW$|;`awCKYQStjMQx-hmzAq#__#xGO`OsNJ$7v8Yu}uNh2jGDCwjm1tp!7l%OO?'
    'NeN2gwvzc}jDM4nOfn)UNm9}TB}Gb_prlC2d3H@=*A&<_iCt4**QDrLLUc`vt|dg*r07~gbWMt`B}CVv=vqQ_EsCxsMAxE`aS0>i'
    'qLFb4BjcixaS0>iqLFb4BjcixaS0>iqLFbIM#e?#+68tkV%ILPYZ1G4fnAH(wFA5AxTk&7$ZOxZ%l(tpqqjGVygqQbfAYx#-)3Zh'
    'k!t6S1Su&&N!(TvuXn7@8p%dR1SLsInxLdeNfVS5De0$`T>144BSSDkN<vW5NJ$7v8YxLZNhc*KDCwl61SLUAN>BpqN`YNL>`H-M'
    'LF`I_T|w+hfn7oDN`YNT>`H-MN$g62T}kXpfn7=LN`YNT>`H-MNg-T92q%Sb2_ak*!X<=oQ3#h1!bKrmLI@Xya0ww?6v8EhaMAEv'
    '!th%({FX5M77f2c7=DX}-ysaYMZ@nnKm4{09N)I{j&B<{zHR3n-!^c3Yv&!`HgJ4v=N;cRaC~cu<BOCuK}nI4(}UKvU*9k?&Km*8'
    'cfs+E9Nz`UH*$Oz9N)<CU2uFS$9KW;RElasDXLK^stKj2My04Gl%g7yqMA^OYE+79LMf_IDXIyjs79rzCX}KYm7<zZifUAfYC<Wh'
    'Q7NhkrKm>h_$I958?ED;u#RuEj&H&`zR^0q!8$%K-@UqN<l{Fb_jkvi9gn_Y<n`l($0p6E4}61>*UdLDvI|B?NeD_BDG5PIBPA&)'
    '>7*nDC7qO%pd?612}<IolFO$r1tpneL{O5XqzOujlr%v}k&@Hwy7ucEMs~pnDG5PIBPAgyX{00tC7qO{prn(M)9ku@qw}_rCcLgj'
    'ud4~KtI_Lf!t3g!qzg(qDd~!mASI^<t)$6a3X{8}$z2MQyQIlo3X{8}$z2MQyQIlo3X{8}$z2MQyQIlo3X{8}$z2MQyQIlo36r~|'
    '$z2JPyQIlo36r~|$z2JPyQIlo36r~`$z2JPyQ0Zm36r~`$z2JPyV@v(+xZZ#jY7Dc58>J<gxgm`xV3Ry(97&!yP((prHy|*zVv<k'
    'wVdzbxD&a5UcmYNf#W23|LlQpGE$w4Ou+~#>8F((Hv~vYpMnum5`vOOMp7`+>2;;>x;iN-K}p;;vVR#F-((~^8M*bkMq9Okk&hqR'
    'nRDcpZyNddL6%P+_%<U2Mru26BuGgKO5(PX*AKFsHj*0|Ijtng@hLby$?+*TKFRSZI6lcIrT8QZj5OhO6)9<gk|HJjw37NV*WNHv'
    '=Z!Qn5`qzkt|3HMQFM*-(N$!m93w?Wz8+okm~bDRIs4a*%->c9$Ir;)Z>%qk{Pw^lx;}Z}+l&+#iS4|RMoL0Z(nv`PN;)Y?K}jbi'
    'B`66}Qi77Wt)zb$<KJW?H!>n9Nm9}TB}Gb_prlAiKdqz_ySl)xPVDLeyE?I}3+(E|t}d{v6T7;=u1@Ug0=qh~s|)Pv#I6w7)rnmp'
    'u&WchLSR=Xc7?#MPV5SST|w*$fn7oD3V~fg><WQhLF@{FT|w*$fn7oD3V~fg><WQhLF`I_T|w+hfn7oDN`YNL>`H-MLF`I_T}kXp'
    'fn7=LN`YNT>`H-MN$g62T}kXpfn7=LN`YNT>`H-MJB9Ergz!!wybB?`QwZ-u2=5fayAZ-Vh43zf@J=DTpAF$lz4qqodVL@TBe!1H'
    'M<aFKNZnS_&W<OqKV#d-NC`$rNeD_BDG5PIBPA&)>7*nDC7qO%pd?612}<I&lG8QFxBU8s5kX0kl6G22C!ch|C!Kr}=Y7(4U{~AE'
    'v#agEuC||NSKDu{Pk!qXdEQ8Yk<;VJ$Htssqy!_RBm^aml!Tz9k&@HnNvAQs3uAnzF}@39e5WzK3uAnzF}^p&C!Lg}prn(M(_?(^'
    '<dZJ=q?1p&;FC^1f&G(;)fO1Zyc`>yxkbHgBrkS8d*B<4)a78`EhCrntAddUMw+0cNJ$fv6e&6HlW{q)2}Y)1gp`D!q>+*klr&P3'
    'f|5>3Qc%)KNeN1Vl$4+(ZY!Bz#`rfG0d`G+U4z&)1$GT$*A&<_h+R`)*CcjLfnAf>H3fD}V%HSdHHlqQVAmveO@UpL*fj-qEsCxs'
    'MAxF|T0(R!imoL@*P`fJLUb*Pt|dg*qUc&mbOk9XK?w|6yD(@i8nkv{&{{NT?N@`=pAIX({`~Oa$M668;m4<+{`&ogr(eGN;~)O*'
    'x8{8p({0w*Kkn3jFCnaV)w7Q;n`8X`K*VPce9|HCA2-s1j4Z(jDM>*|Cnb4WNhhCl!6%)RbVW%gC3RZKQP#e#B)<GDzRk!2BOxeh'
    'q$C6-aa&1#`98nNNCp`Zlq4x>f|4R7O;A##q@PxjlqV_VNm8DqkS9rbl0u#&jVCFLCrK1efx<}?PJzN#etpA;paj@e0=tsfRRX(`'
    '*i{0%lGs%OyOP*d0=tUXRRX(;*i{0%ir7^GyNcLV0=tUXRRX(;*i{0%ir7^GyNdE;2zfFnPlk{ugYskuc`_(ZhL9(N@?;2kGAK`m'
    'kSBxkWC(dOC{KovCxgcLDU9)h#`r0W@q@<rDU9)hurdX#Ov1_(urdiNQ^3k3tV{tbldv)ctW3hn6tFT0EAu>7qQf*b;`}uAVq}4l'
    '(|Hn9iVC3=6;z4}p%fKViVC3=6;vz;p;!=9EQnN;G*Xg+l1@reP|``s`HZLKTnNj#pygZ$%ekQCTnNj#+#tG+2Tn)V@iRIZDZ$9?'
    '8RW%?@Vai#AhVsHL0-S|H;f2Ml9V(-Ns*D$BV*F=o5Jv$LD3aLbkY7v3i~HX`zI;vpCs*{q_BUIw6&4K)<)9SMhaUSS>*U3IG&0H'
    'DHIEmiUlbY3zCWjDHIEm*6}H<<CE6$DXinKU-=tG1SLgE&L<-k3sNW+<Rr&0!SRzEzXZoma{Lk;zeve0DA{4ced%u?7-?kW^jx@T'
    'E?mM~xM(h1!d$p$np(m%wHjpn<>QNnkp)IhM_18FwGvLMT|P{ATS*r}n5sl2REdfTIF*VK@QF~sDVh?MFeNIQ5|uC|D%w9OVgICP'
    '-c`c9t7zU;!n~_!-c`c9t7zU;!n~_!-c>i|UH@pqk~X1XNn7W8?p%!QVB~bqo#SV0q~vtZor@7t5`vOON<vW5NJ$DxIw?s(Nhc*G'
    'C<#(hf|A?)!rFemU)Z%@-!LL5Nm9}TB}Gb_prlC2d3I5+!zT1PY}D(p3B3**^*U@qufs;Y4x7;Huu-qWCiFUN)a$Sby$&1oI&4C('
    '!$!RhyU^>fQLn=;^g3+R>#z&G4jc74>_V@@M!gQZ(Ce^Mufs0%I_%WzunWBoJM}v3La)P4jjX%S$huP_>n=31?$pS-3yrL={rZLx'
    'K?$&{oo834CYN1ka@ncLWfz)Uc4~6j+j(|f`}Nw$d3H5oR}<LPh+R!!S0i>cfnC&9z6)LDJ9U-sLRb0gSAP9K8Afiut}bAu5mvf@'
    'l}1?U0#+Jfr3+YTgq1E}r4v@VfR#>I=>k?dVWkUL>4cRoV5JjQx`35VQ=%?Ri8`^X3+(E|t`OMOiCrPEs}s9IU{@!0g}|;(><WQh'
    'K_MJM2nU652q7F4!YPDsPza|G!a*UNLI?+ia0($D6v8QlaMJLb!tk3k{H8GcCJnzS48KXkZwkY2((s$Y@SDW06xfx-t`yjn#I6$9'
    'mBg+R*p<Yt64;f*t`gXl#I6$9Rm83m*j2=?64+J5t`gW)#I6$9Rm83m*j2=?64+J5t`gW)#I7N*tB74gVAr4!9zqBY3gIDy@SqSL'
    'LI@8E;UR?Zpb#EH2oDP3DTMH#5S~g1Cn;%yk|HHdP*SAi{O~(z_?^P=J8Afx!tgtZT~lD!Bz8@KU6a@~1$NQt!7iL0>~wmt3#SJ='
    'ogVDM>A?tae8hRjM}Xra&O1J69UsCvK4={u!a6=^9UsCvK4={u!a6=^1{uN(GH3=F!VEGBhTj<Hhu<g|eq)>;exqRcjd6bXz4q%H'
    'Ms~pnDG5PIBPAgyX{00tC7qO{prn(M5|jifDM1OaOISJtEgeEwIs`2pLRdNkEgeEwIs`2pLRdNkEgeEwIs`2pLRdNkEgeEwIs`2p'
    'LRdNkEgeEwI$Znp`hgOZ1Su&&39xGk?3%=`C9rD}yOzK%x;q!b-MOH<b0OTF3tIh#u=)*J{f4mm4O;z%vHGQpM<HB13c7d{!o{PY'
    'i$@_`JPNva6vD-$po>Q#Ts#W8cof3Lqol+8DIDHUI=r94;r*n;`zajWPddDx!r}d-!}}>5-cLHbpTgn&q{I6u9Ntelyr071{iMVD'
    'DIDHUI=r94;r*n;`zajWPddDx!r}d-!}}>5-oN(i^#dg+2~tvm5@45bLm=sfKngbml5Pm3a6=&JhCm881d?tDq;NwZ>4rcGHw2Py'
    '2&8aBAnArc3O59jZV04sLm=sfKngbml5Pm3a6=&JhCm881d?tDq;NwZ>4rcGHw2Py2&8aBAnArc3O59jZV04sL*Ux4Zy4DHBcvn*'
    'C5@DXprnzK6qIyQl7f;>N=i@?q@)BTz%JoxNYcrI6iyZ-oh(Q=Sx{}S?WU2B%~I=Ki`izpVdP^E((SVczR5`2$VdrBNJ$DxIw?s('
    'Nhc*GC<#(hf|9tc<VYkzNp56BP?Dsi2}+8TG(ky`l73prm0#a5Qi2gu5`vOON<vW5NJ$DxIw?s(Nhc*GC<#(hf)Zd?3G7N@R|)J&'
    'Vpj?5N@7<D>`G!+3G6CjR|)JYVpj?5Dq>d&>?&ec3G6CjR|)JYVpj?5Dq>d&>?&f{5ZG13t|73ih+RWqR}s60z^)>84S`*Q*fj)p'
    '4Pw_2*foe<Ltxh+b`61DgV;3$b`4_J5ZE<{T|;2kAa+fGU4z&)1$GT$*A&<_h+R`)*C2LHfnAf>H3fD}V%HSdHHlqQVAmveO@UpL'
    '*fj-qO=8y+*foh=Q()I5b}fNjli0Ncc1>c}64*6~T}xosBz7%<U5nVY1a>WA*Am#Zh+Ruy*CKW;fnAH(wFGu8V%HMbwTN9yVAmpc'
    '?E<?Nv1=FDwNnW1LJ02^!n+W{JB9Ergz!!wybB?`QwZ-u2=5TWu?Zo(LkM4t2uhNaG(ky`k|roAQgV9uz4q%HM(VtgAR}cMx&6A@'
    '`4B#gKnS<<A$<IdMoLZt^7t8@l%$}fladmY1Su&&2}D;HqN`DKbtSrzlr%v}k&-4TDN=GCke#Bd3(?gnx<ZJqPSF)YbajfZ5TdJ7'
    'bcGOIouVs*=n5KsLl}PPEKvw&iGt1&g>aTA=qynPXNiK&5`}P<DCjIv2xp0c&Ju-imMG{fQ3z*=f(~MYa1bMCGc$zE%%IK85H>S|'
    'HZw!m%naJh3}G`fXfrc}&CH<9%n&v+gElim*vt&t2?}8+C}<}rgq@(EouCkQf`V%GA=K)FYV{%1>Vs<aA=K)FYV{%1>Vs<aA=K)F'
    'YV{%1>Vs<aA=K)FYV{%1>Vs<aA=K)FYV{%1>Vs<aA=K)FYV{%1>aYEJZKMfGij*`#Ns*HC?AnQ4yTGoU*tHAnqD`6*Hfe%3X+qef'
    '3EHFyVUs3klO}{snxIXZ5H@LoHfd7WqzT%jNnw*FXp<&|O`4!hniMu^f;MSV*rZ9?q)B0u=JKJP+e+GbpCr}lQ>fJ^)#_8I)hE^J'
    'Q>fJ^)#|e=J}FYt1SLgE&gV&|Jn2H7(1t(?8v;oi0x4_=Bwfi(;Yx1OmE07r<R)FoP2oyzQW-r{@d;o>D5EEp(Nid+Cza7tD5EEp'
    '(Nid+Cza7tD5EEp(Nid+Ctb-+;Yx1OEx;6R0Vdr7OyL$_(k;L&<#jbul7f;>N>WhLNy%x{7fn-3n5G^^ZnLY}d3MzX?5cL2UDY5N'
    'tDR5AYLJZ8&L`uwU*9kyC~2f51tp!7q@bjelJm*f$tPX#NhhCl!6%)3(gmM%@<~^G5~QRACAa6oC9F@14lI>$V5#W9QV9o^iViH5'
    'aA2wEz)}eZmWmE6m2hCG=+jyyd|IpM(^@5bTC3>OS|xm1t7sixUth<!Hji(rMQmUDrds=#Hh=q8-u7<h@i*3IMt*<bbpi9~1K(z('
    'z)0K98!1xK1SLgE`e`Mv3sW#sf)P>@f|5o`LQv93NeW6jDM>*|CnY5)2~tvmlDMs;zKrp2G6L)>fn7=LDuG=|>?(m>N$e_tT}A9F'
    'fn7!HDuG=^>?(m>MeHhpT}A9Ffn7!HDuG=^>?(m>MeG^^yNcK~1a=j%YY6NrV%HGZRm83#uxk*zhQO{t>>2{Q2C-`h>>9+bA+T!@'
    'yN1B7LF^g=y9Tjq2<#fft|_o<5WA+pu0iaY0=ovWYYOZd#I7l@YZAMrz^+N`ngY8fv1<zKn#8Utuxk>#rogUA?3x0*Cb4S@?3%=`'
    'C9rD}yOzMNN$gqzyC$(~3GAB0t|hQ*5xbVau0`xx0=pKmYYFUH#I7Z<YZ1Gaz^+B?S^~Qkv1<wJTEwnhVAmpc?E<?Nv1=FDwTNB2'
    'z^+B?+68v)#I9Xn*G}x(1$OPku3cc)PVCwRcJ0KjU0~Nv?Air(?ZmEKVAl@ps!d?m4(zH;VAl@ps!d?m4(zH;VAl@ps_i_xuKoIk'
    'krIrMk`R<MQWAoaMoLbztEetqLS4A1E?h!gxTr2%LS4A1E?h!gxTr2%LS4A1E?h!gxTr2%LS4A1E?h!gxTq9WLMf`K6jeegs;CrI'
    'Q+ZvTl%${pl2IrY6cr0fC>9hI3+i;S;Ml^xeJbjB;Pk1eS0lG5{Lx4WMo39Nt>pHpsE<bKyb(}Rf|4R7B`CRlM&k8>5R^1h5`vOO'
    'N>WhLNl6MyIw>hZNsy8fl*DZ%r_V^d<<~cioK`Z(CqwYbAfF7uCzFzK3duMr8K;nplag@?$vDaJQ*itw$4|xakc>-6#!1Pzgk+pF'
    'C0fFiXwsBu2~(m)j$eY~7dd_x9KXo%yWsdmj^72x@9?_XCcLg4URT?M*R{jzYMb)9ij<trlePhc+jgG9ZG&WN+xcW{8%0+;A6>1%'
    '$k^KXk+C)MNfUh1$R|zkNh6<h#V3&Q{d~q>`}NvL6O<GwX@Zg>C8w9zEnu43;`}tV1tnt$$(S&!Z+U)J-;!Qe3a^XS@vRgi;1gjT'
    '-)J4*gmrwQb$k=n@r~B;O<2b_TE{nG9p8!oIRro!0dfd{ECS>Z06Cxr*~a-AWE)U}Y~y?lvJC>{5CA!dT|;2kAa)IbT~zOCLcOa|'
    'y{ifJt~Q8WQ()I1c1?j@gV;3%b`4_J6xcP0T~lD!Bz8@KU6a@~1$I%%wh1NMMkU)Olx!Q7Y@1NBZB(*tLdmvG3gPQr>^~h={_^m{'
    'ho^^MKm7FL_kaEH<I_)n{r<z#FW>#~5C8VbXFu}zU%&a;kB`^1_jkvi9gn{8*^iGmZS%7SKL57JKV;;x)k#Kn!3Ze{K}jPeAt-62'
    'Bn2g%l%$}fladmY1Su&&N!(U4zKrp2GLlI~1SLsInxLdeNjt4%PzVnpga?K25JGrR2oE8I2ZityLU>RJPa%W{h42(YcoK!DK;cOt'
    'JcSUR6v9&o;YlGpg%Dme{I1)>??3bfTI#WuLXWi^^qJcse9?T+7tM$8Me{*lG@rs3%?Ewad<tJQAM{1@DSXj<&=<|8@I~`MUo@Y>'
    '7tN_nU<z#lliCEP&?YdcO<)Rb0<Zmg{Xhyz!0|!{zoZU+DRl5l>fo0`2fw5aekpYDOX}d4LI=O34t^<g@Js68mqG`>qz-;5bnr`R'
    'f0jb~v!wQCDYQRJYJZkO`?I75RV6g2Dr!(wLW8QJ<`pG0uc)LC_NMT`-lPxqrtq=xq>qiK@Ud}fUQt5xilXKfB{Z)nYF<&LysjW6'
    'B`AS!vk&3h>_y*ZAHuiUi@wc1gm1G`*Y^^-z87_UFQMyuQP=nSde`@V_{Lk%h4U0HoF`p4PvOFO(uMOBE}SP_I8WiidD4aR6fT@!'
    '`}GYYf|5>3N>CD{qy#1Kp`RvvEMub&{WRfY85@1*rwJd+*yuw)P54;GM(k<=yBe{p3G8abt|qXn5xbheu14%?0=pWqs|oCC#I7c='
    's}Z}Jz^+E@>H@nOv8xO0YQ(NCu&WWfy1=eR?CJu$I<cz@?CQj>F0iW;ySl)xPVDLeyE?I}3+(E|t}d{v6T7;=u1@R<fnA;06#~0D'
    'u`2|2bz)Zt?CQj>xIQWOxBm}*osMq'
)
ABLATION_SNAPSHOT = json.loads(zlib.decompress(base64.b85decode(ABLATION_SNAPSHOT_PAYLOAD)))

## Pemeriksaan sebelum fitting

Periksa hash CSV, waktu/angka baseline, dukungan interval, grup orang, larangan
ID Test, serta cakupan fold. Tidak ada training penelitian pada tahap ini.

In [ ]:
from IPython.display import display

prepared = prepare_ablation(DATASET_ROOT, ABLATION_SNAPSHOT)
print('Train:', len(prepared['train']), 'window /', prepared['train'].participant_id.nunique(), 'orang')
print('Validation:', len(prepared['validation']), 'window')
for name in ('train', 'validation'):
    print(name, 'support kelas:')
    display(prepared[name].cognitive_load_label.value_counts().reindex(LABELS, fill_value=0))
print('Fold:', {key: len(value) for key, value in prepared['folds'].items()})
print('PERHATIAN: validation tanpa Medium tidak menilai tiga kelas secara lengkap.')

## Jalankan ablasi

Budget tetap 45 fitting CV: empat varian Extra Trees dan satu Dummy pada
sembilan fold. Empat fitting Train penuh mengevaluasi semua varian yang telah
dibakukan pada validation historis. Tidak ada pemenang yang dipilih berdasarkan
validation, refit Train + validation, atau evaluasi Test. Setiap Run all membuat
folder run baru dan mempertahankan hasil lama.

In [ ]:
training = run_ablation(DATASET_ROOT, OUTPUT_ROOT, ABLATION_SNAPSHOT,
                        ABLATION_IMPLEMENTATION_HASHES)

## Hasil

Utamakan OOF tiga kelas pada orang yang ditahan dari fitting. Baca kontras
eye-only vs raw dan gabungan vs RMSSD-only untuk memisahkan manfaat transformasi.
Laporkan juga metrik berbobot sama per orang dan jumlah sumber. Semua perbandingan
dihitung dari hasil Exp02; skor Exp01 hanya menjadi pemeriksaan kontrol.

Empat peserta membatasi kesimpulan. Label masih difficulty/fase, acuan eye masih
pre_task, dan window overlap bukan observasi independen. Angka validation tidak
boleh dipromosikan menjadi keberhasilan tiga kelas pada pengguna baru.

In [ ]:
from IPython.display import Image, Markdown, display

display(training['leaderboard'])
display(training['contrasts'])
display(training['validation'])
display(training['controls'])
display(Markdown((training['run_dir'] / 'RESULTS.md').read_text(encoding='utf-8')))
for name in ('ablation_oof.png', 'ablation_contrasts.png', 'validation_support.png'):
    display(Image(filename=str(training['run_dir'] / 'figures' / name)))
print('Hasil:', training['run_dir'])